In [ ]:
import json
import numpy as np
import pandas as pd

# Colab: mount Google Drive (safe to skip outside Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# Match IDPInterp config
JSON_PATH = "/content/drive/My Drive/Colab Notebooks/DisProt release_2025_12 with_ambiguous_evidences.json"
SEED = 42
MAX_PROTEINS = 2000
MIN_LEN, MAX_LEN = 20, 1200
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

def extract_entries(raw):
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        for k in ["data", "entries", "proteins"]:
            if isinstance(raw.get(k), list):
                return raw[k]
        return [raw]
    raise ValueError("Unsupported JSON format")

def get_acc(entry, i):
    for k in ["acc", "accession", "disprot_id", "id"]:
        if isinstance(entry.get(k), str):
            return entry[k]
    return f"unknown_{i}"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

entries = extract_entries(raw)

rows = []
for i, e in enumerate(entries):
    seq = e.get("sequence")
    if not isinstance(seq, str) or not seq.strip():
        continue
    seq = seq.strip().upper()
    L = len(seq)
    if L < MIN_LEN or L > MAX_LEN:
        continue
    rows.append({"acc": get_acc(e, i), "sequence": seq, "length": L})
    if len(rows) >= MAX_PROTEINS:
        break

manifest = pd.DataFrame(rows).drop_duplicates(subset=["acc", "sequence"]).reset_index(drop=True)

# Trevor-style split
rng = np.random.default_rng(SEED)
idx = np.arange(len(manifest))
rng.shuffle(idx)

n = len(manifest)
n_train = int(n * TRAIN_FRAC)
n_val = int(n * VAL_FRAC)

split = np.array(["test"] * n, dtype=object)
split[idx[:n_train]] = "train"
split[idx[n_train:n_train+n_val]] = "val"

manifest["split"] = split
manifest["row_id"] = np.arange(len(manifest))

manifest.to_csv("protein_manifest_idpinterp.csv", index=False)

print("Saved protein_manifest_idpinterp.csv")
print(manifest["split"].value_counts())
display(manifest.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved protein_manifest_idpinterp.csv
split
train    1400
val       300
test      300
Name: count, dtype: int64


,acc,sequence,length,split,row_id
0,P03265,MASREEEQRETTPERGRGAARRPPTMEDVSSPSPSPPPPRAPPKKR...,529,train,0
1,P49913,MKTQRDGHSLGRWSLVLLLLGLVMPLAIIAQVLSYKEAVLRAIDGI...,170,train,1
2,P03045,MDAQTRRRERRAEKQAQWKAANPLLVGVSAKPVNRPILSLNRKPKS...,107,train,2
3,P00004,MGDVEKGKKIFVQKCAQCHTVEKGGKHKTGPNLHGLFGRKTGQAPG...,105,val,3
4,P27695,MPKRGKKGAVAEDGDELRTEPEAKKSKTAAKKNDKEAAGEGPALYE...,318,train,4


In [ ]:
# Step 2: build protein-level labels for one feature (start with disorder)

MANIFEST_PATH = "protein_manifest_idpinterp.csv"
TERM = "disorder"  # start simple

manifest = pd.read_csv(MANIFEST_PATH)

with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)


def canonical_term(x):
    x = str(x).strip().lower().replace("/", " ").replace("-", " ")
    x = " ".join(x.split()).replace(" ", "_")
    return x


def get_regions(entry):
    # Same spirit as IDPInterp: try common region containers.
    if isinstance(entry.get("regions"), list):
        return entry["regions"]
    if isinstance(entry.get("disprots"), list):
        return entry["disprots"]
    return []


def get_span(region):
    if "start" in region and "end" in region:
        return int(region["start"]), int(region["end"])
    for a, b in [("begin", "end"), ("from", "to")]:
        if a in region and b in region:
            return int(region[a]), int(region[b])
    return None


def get_region_term(region):
    for k in ["term_name", "name", "type", "term"]:
        if isinstance(region.get(k), str):
            return canonical_term(region[k])
    anns = region.get("annotations")
    if isinstance(anns, list):
        for ann in anns:
            if isinstance(ann, dict):
                for k in ["term_name", "name", "type", "term"]:
                    if isinstance(ann.get(k), str):
                        return canonical_term(ann[k])
    return "unknown"


entries = extract_entries(raw)

# Build accession lookup for quick join manifest -> annotations.
entry_map = {}
for i, e in enumerate(entries):
    acc = get_acc(e, i)
    seq = e.get("sequence")
    if isinstance(seq, str) and seq.strip():
        entry_map[acc] = e

rows = []
for _, r in manifest.iterrows():
    acc = r["acc"]
    seq = r["sequence"]
    L = len(seq)

    e = entry_map.get(acc)
    if e is None:
        rows.append({
            "acc": acc,
            "split": r["split"],
            "length": L,
            "term": TERM,
            "positive_residues": 0,
            "protein_score": 0.0,
            "protein_label": 0,
        })
        continue

    mask = np.zeros(L, dtype=np.uint8)
    for region in get_regions(e):
        if not isinstance(region, dict):
            continue

        # Trevor-style behavior for disorder: include all annotated regions.
        if TERM != "disorder":
            if get_region_term(region) != TERM:
                continue

        span = get_span(region)
        if span is None:
            continue

        s, t = span  # DisProt spans are 1-indexed inclusive
        s = max(1, s)
        t = min(L, t)
        if t >= s:
            mask[s - 1 : t] = 1

    pos = int(mask.sum())
    score = float(pos / L) if L > 0 else 0.0

    rows.append({
        "acc": acc,
        "split": r["split"],
        "length": L,
        "term": TERM,
        "positive_residues": pos,
        "protein_score": score,
    })

labels_df = pd.DataFrame(rows)

# Disorder fix: use high-vs-low quantile labeling instead of any-positive.
# This avoids the degenerate all-positive class issue on DisProt.
if TERM == "disorder":
    LOW_Q = 0.33
    HIGH_Q = 0.67
    q_low = labels_df["protein_score"].quantile(LOW_Q)
    q_high = labels_df["protein_score"].quantile(HIGH_Q)

    labels_df["protein_label"] = -1  # middle band (unused for binary probe)
    labels_df.loc[labels_df["protein_score"] <= q_low, "protein_label"] = 0
    labels_df.loc[labels_df["protein_score"] >= q_high, "protein_label"] = 1

    print(f"Disorder labeling mode: quantile binary (low<={q_low:.3f}, high>={q_high:.3f})")
else:
    labels_df["protein_label"] = (labels_df["protein_score"] > 0.0).astype(int)

labels_df.to_csv("labels_disorder.csv", index=False)

print("Saved labels_disorder.csv")
print("Counts by split and label (-1 means dropped middle):")
print(labels_df.groupby("split")["protein_label"].value_counts(dropna=False))
display(labels_df.head())

Disorder labeling mode: quantile binary (low<=0.105, high>=0.315)
Saved labels_disorder.csv
Counts by split and label (-1 means dropped middle):
split  protein_label
test    0               104
       -1                98
        1                98
train  -1               497
        1               452
        0               451
val     1               110
        0               105
       -1                85
Name: count, dtype: int64


,acc,split,length,term,positive_residues,protein_score,protein_label
0,P03265,train,529,disorder,52,0.098299,0
1,P49913,train,170,disorder,37,0.217647,-1
2,P03045,train,107,disorder,107,1.000000,1
3,P00004,val,105,disorder,105,1.000000,1
4,P27695,train,318,disorder,43,0.135220,-1


In [ ]:
# Step 3A: build/reuse ESM protein embeddings for manifest proteins

# If running in Colab for the first time, uncomment:
# !pip -q install transformers scikit-learn

import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel

EMBED_CACHE = "embeddings_idpinterp.npz"
MODEL_NAME = "facebook/esm2_t12_35M_UR50D"
BATCH_SIZE = 32
MAX_TOKENS = 256

manifest = pd.read_csv("protein_manifest_idpinterp.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

if os.path.exists(EMBED_CACHE):
    data = np.load(EMBED_CACHE, allow_pickle=True)
    X = data["X"]
    accs = data["accs"]
    print("Loaded cached embeddings:", X.shape)
else:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)
    model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    model.eval()

    seqs = manifest["sequence"].tolist()
    accs = manifest["acc"].astype(str).to_numpy()

    @torch.no_grad()
    def embed_batch(batch_seqs):
        enc = tok(
            batch_seqs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_TOKENS,
        ).to(device)
        out = model(**enc).last_hidden_state
        attn = enc["attention_mask"].unsqueeze(-1)
        pooled = (out * attn).sum(dim=1) / attn.sum(dim=1).clamp(min=1)
        return pooled.detach().cpu().numpy()

    chunks = []
    for i in range(0, len(seqs), BATCH_SIZE):
        chunks.append(embed_batch(seqs[i:i+BATCH_SIZE]))
        if (i // BATCH_SIZE) % 20 == 0:
            print(f"embedded {min(i + BATCH_SIZE, len(seqs))}/{len(seqs)}")

    X = np.vstack(chunks)
    np.savez(EMBED_CACHE, X=X, accs=accs)
    print("Saved:", EMBED_CACHE, "shape:", X.shape)

# Keep a dataframe for safe alignment merges later
embed_df = pd.DataFrame({"acc": accs})
embed_df["embedding"] = list(X)
print(embed_df.head(2))

device: cuda
Loaded cached embeddings: (2000, 480)
      acc                                          embedding
0  P03265  [0.016788822, -0.08905204, 0.11723822, -0.0217...
1  P49913  [-0.049455006, -0.053591173, 0.10413048, 0.076...


In [ ]:
# Step 3B: train first probe (disorder) and evaluate by split

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

labels_df = pd.read_csv("labels_disorder.csv")
manifest = pd.read_csv("protein_manifest_idpinterp.csv")

# Join manifest + labels + embeddings by accession to avoid row-order mistakes
work = manifest[["acc", "split"]].merge(
    labels_df[["acc", "protein_label", "protein_score"]], on="acc", how="inner"
)
work = work.merge(embed_df, on="acc", how="inner")

# Keep only binary-labeled rows (0/1). For disorder we drop middle-band -1 rows.
work = work[work["protein_label"].isin([0, 1])].copy()

print("rows after merge/filter:", len(work), "(manifest:", len(manifest), ")")

X_all = np.stack(work["embedding"].to_list(), axis=0)
y_all = work["protein_label"].astype(int).to_numpy()

train_idx = work["split"].values == "train"
val_idx = work["split"].values == "val"
test_idx = work["split"].values == "test"

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val, y_val = X_all[val_idx], y_all[val_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]

print("train/val/test:", X_train.shape, X_val.shape, X_test.shape)
print("label balance train:", np.bincount(y_train))

if len(np.unique(y_train)) < 2:
    raise ValueError(
        f"Train split has one class only: {np.unique(y_train)}. "
        "Adjust label rule/quantiles or choose another term."
    )

clf = LogisticRegression(max_iter=3000, class_weight="balanced")
clf.fit(X_train, y_train)

def eval_split(name, Xs, ys):
    p = clf.predict_proba(Xs)[:, 1]
    pred = (p >= 0.5).astype(int)
    out = {
        "split": name,
        "acc": accuracy_score(ys, pred),
        "f1": f1_score(ys, pred, zero_division=0),
        "auc": roc_auc_score(ys, p) if len(np.unique(ys)) > 1 else np.nan,
    }
    print(out)
    return out

val_metrics = eval_split("val", X_val, y_val)
test_metrics = eval_split("test", X_test, y_test)

# Save simple artifacts for next steps
np.savez(
    "probe_disorder_data.npz",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test,
)
print("Saved probe_disorder_data.npz")

rows after merge/filter: 1320 (manifest: 2000 )
train/val/test: (903, 480) (215, 480) (202, 480)
label balance train: [451 452]
{'split': 'val', 'acc': 0.7627906976744186, 'f1': 0.7627906976744186, 'auc': np.float64(0.8437229437229438)}
{'split': 'test', 'acc': 0.8118811881188119, 'f1': 0.8041237113402062, 'auc': np.float64(0.86175431711146)}
Saved probe_disorder_data.npz


In [ ]:
# Step 4A: build/reuse a token-level SAE on ESM token activations

# If needed in Colab:
# !pip -q install transformers

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "facebook/esm2_t12_35M_UR50D"
MAXLEN = 256
SEQ_TRUNC = 200
K_TOKENS = 60
N_PROT_FOR_SAE = 1000  # keep manageable for Colab
D_LATENT = 2048
SAE_EPOCHS = 8
SAE_BS = 4096
SAE_LAMBDA = 1e-3

TOKEN_ACTS_CACHE = "train_token_acts_disorder.npy"
SAE_CACHE = "sae_disorder.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Always load tokenizer/model here because later steps need token-level access.
tok = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

manifest = pd.read_csv("protein_manifest_idpinterp.csv")
train_manifest = manifest[manifest["split"] == "train"].copy().reset_index(drop=True)
train_seqs_for_sae = [s[:SEQ_TRUNC] for s in train_manifest["sequence"].tolist()[:N_PROT_FOR_SAE]]

@torch.no_grad()
def collect_token_acts(seqs, batch_size=16, K=60, max_length=256):
    h_list = []
    for i in range(0, len(seqs), batch_size):
        batch = seqs[i:i+batch_size]
        enc = tok(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(device)
        out = model(**enc).last_hidden_state
        attn = enc["attention_mask"].bool()

        for b in range(out.shape[0]):
            hb = out[b][attn[b]]
            # Drop special tokens for cleaner residue-level semantics
            ids = enc["input_ids"][b][attn[b]]
            special_ids = set(tok.all_special_ids)
            keep = torch.tensor([tid.item() not in special_ids for tid in ids], device=hb.device)
            hb = hb[keep]
            hb = hb[:K]
            if hb.shape[0] > 0:
                h_list.append(hb.detach().cpu().numpy())

        if (i // batch_size) % 20 == 0:
            print(f"token acts: {min(i + batch_size, len(seqs))}/{len(seqs)}")

    return np.concatenate(h_list, axis=0)

if os.path.exists(TOKEN_ACTS_CACHE):
    H = np.load(TOKEN_ACTS_CACHE)
    print("Loaded token activation cache:", H.shape)
else:
    H = collect_token_acts(train_seqs_for_sae, batch_size=16, K=K_TOKENS, max_length=MAXLEN)
    np.save(TOKEN_ACTS_CACHE, H)
    print("Saved token activation cache:", H.shape)

d_model = H.shape[1]
print("d_model:", d_model)

class SAE(nn.Module):
    def __init__(self, d_in, d_latent):
        super().__init__()
        self.enc = nn.Linear(d_in, d_latent)
        self.dec = nn.Linear(d_latent, d_in)

    def forward(self, x):
        z = torch.relu(self.enc(x))
        xhat = self.dec(z)
        return xhat, z

sae = SAE(d_model, D_LATENT).to(device)

if os.path.exists(SAE_CACHE):
    sae.load_state_dict(torch.load(SAE_CACHE, map_location=device))
    print("Loaded SAE from cache")
else:
    H_t = torch.tensor(H, dtype=torch.float32)
    dl = DataLoader(TensorDataset(H_t), batch_size=SAE_BS, shuffle=True, drop_last=True)
    opt = torch.optim.AdamW(sae.parameters(), lr=1e-3)

    for ep in range(SAE_EPOCHS):
        sae.train()
        total_mse, total_l1 = 0.0, 0.0
        for (x,) in dl:
            x = x.to(device)
            xhat, z = sae(x)
            mse = ((xhat - x) ** 2).mean()
            l1 = z.abs().mean()
            loss = mse + SAE_LAMBDA * l1
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_mse += mse.item()
            total_l1 += l1.item()
        print(f"epoch {ep+1}/{SAE_EPOCHS} mse={total_mse/len(dl):.6f} l1={total_l1/len(dl):.6f}")

    torch.save(sae.state_dict(), SAE_CACHE)
    print("Saved SAE to", SAE_CACHE)

sae.eval()
print("SAE ready. latent dim:", D_LATENT)

device: cuda


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


token acts: 16/1000
token acts: 336/1000
token acts: 656/1000
token acts: 976/1000
Saved token activation cache: (59816, 480)
d_model: 480
epoch 1/8 mse=0.067186 l1=0.106285
epoch 2/8 mse=0.027857 l1=0.126695
epoch 3/8 mse=0.017568 l1=0.138263
epoch 4/8 mse=0.011405 l1=0.161530
epoch 5/8 mse=0.007574 l1=0.190061
epoch 6/8 mse=0.005283 l1=0.211585
epoch 7/8 mse=0.003838 l1=0.224511
epoch 8/8 mse=0.002847 l1=0.232942
Saved SAE to sae_disorder.pt
SAE ready. latent dim: 2048


In [ ]:
# Step 4B: rank disorder-associated latents on train split

import numpy as np

LATENT_RANK_CACHE = "latent_rank_disorder.npz"
M_FOR_RANK = 800

labels_df = pd.read_csv("labels_disorder.csv")
manifest = pd.read_csv("protein_manifest_idpinterp.csv")

train_labeled = manifest.merge(
    labels_df[["acc", "protein_label"]], on="acc", how="inner"
)
train_labeled = train_labeled[
    (train_labeled["split"] == "train") & (train_labeled["protein_label"].isin([0, 1]))
].copy().reset_index(drop=True)

@torch.no_grad()
def get_token_hidden(seq, tok, model, max_length=256, drop_special=True):
    enc = tok([seq[:SEQ_TRUNC]], return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc).last_hidden_state[0]
    ids = enc["input_ids"][0]
    attn = enc["attention_mask"][0].bool()
    h = out[attn]
    ids = ids[attn]

    if drop_special:
        special_ids = set(tok.all_special_ids)
        keep = torch.tensor([tid.item() not in special_ids for tid in ids], device=h.device)
        h = h[keep]
    return h

@torch.no_grad()
def protein_latent_summary(seq, tok, model, sae, reduce="max"):
    h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
    h = h[:K_TOKENS]
    if h.shape[0] == 0:
        return np.zeros(sae.enc.out_features, dtype=np.float32)
    z = torch.relu(sae.enc(h))
    if reduce == "max":
        Z = z.max(dim=0).values
    elif reduce == "mean":
        Z = z.mean(dim=0)
    else:
        raise ValueError("reduce must be 'max' or 'mean'")
    return Z.detach().cpu().numpy().astype(np.float32)

def rank_latents_by_correlation(Zmat, y):
    Zc = (Zmat - Zmat.mean(axis=0)) / (Zmat.std(axis=0) + 1e-8)
    yc = (y - y.mean()) / (y.std() + 1e-8)
    corr = (Zc * yc[:, None]).mean(axis=0)
    ranked = np.argsort(-np.abs(corr))
    return ranked, corr

if os.path.exists(LATENT_RANK_CACHE):
    cache = np.load(LATENT_RANK_CACHE)
    ranked_latents = cache["ranked_latents"]
    corr = cache["corr"]
    alphas = cache["alphas"]
    print("Loaded latent ranking cache")
else:
    M = min(M_FOR_RANK, len(train_labeled))
    seqsM = [s[:SEQ_TRUNC] for s in train_labeled["sequence"].tolist()[:M]]
    yM = train_labeled["protein_label"].to_numpy(dtype=np.float32)[:M]

    Zmat = np.stack([
        protein_latent_summary(seq, tok, model, sae, reduce="max")
        for seq in seqsM
    ], axis=0)

    ranked_latents, corr = rank_latents_by_correlation(Zmat, yM)

    top1_latent = int(ranked_latents[0])
    top1_sd = Zmat[:, top1_latent].std() + 1e-8
    alphas = np.array([0, 1*top1_sd, 2*top1_sd, 4*top1_sd], dtype=np.float32)

    np.savez(
        LATENT_RANK_CACHE,
        ranked_latents=ranked_latents,
        corr=corr,
        alphas=alphas,
    )
    print("Saved latent ranking cache")

top20 = ranked_latents[:20]
print("Top latent indices:", top20)
print("Top correlations:", corr[top20])
print("Alpha grid:", alphas)

Saved latent ranking cache
Top latent indices: [ 379  645   50 1338 1012  285  983  169 1536 1688 1543 1168  900 1127
  543  958 1738  946  386 1224]
Top correlations: [-0.2769667  -0.27348617 -0.26234847 -0.26010007 -0.25544    -0.251821
 -0.25139973 -0.24958958  0.2461     -0.2448317  -0.24417618 -0.24314995
 -0.24025725 -0.23445483 -0.23200977 -0.23127855  0.23086996 -0.23048505
 -0.23014341 -0.22707897]
Alpha grid: [0.         0.13594407 0.27188814 0.5437763 ]


In [ ]:
# Step 5: first latent intervention test (single example + small batch)

import numpy as np
import pandas as pd
import torch

EDIT_FIRST_K = 30
K_LATENTS_LIST = (1, 5)
ALPHAS_TO_TRY = alphas[:3]  # keep runtime short

@torch.no_grad()
def seq_to_original_embedding(seq):
    h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
    h = h[:K_TOKENS]
    if h.shape[0] == 0:
        # fallback vector if sequence has no valid tokens after filtering
        return np.zeros(480, dtype=np.float32)
    return h.mean(dim=0).detach().cpu().numpy().astype(np.float32)

@torch.no_grad()
def edit_sequence_embedding(seq, latent_indices, alpha):
    h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
    h = h[:K_TOKENS]
    if h.shape[0] == 0:
        return seq_to_original_embedding(seq)

    z = torch.relu(sae.enc(h))
    z_edit = z.clone()
    k = min(EDIT_FIRST_K, z_edit.shape[0])
    z_edit[:k, latent_indices] += float(alpha)
    h_hat = sae.dec(z_edit)
    x_hat = h_hat.mean(dim=0).detach().cpu().numpy().astype(np.float32)
    return x_hat

def predict_prob(x):
    return float(clf.predict_proba(x.reshape(1, -1))[0, 1])

labels_df = pd.read_csv("labels_disorder.csv")
manifest = pd.read_csv("protein_manifest_idpinterp.csv")

test_eval = manifest.merge(labels_df[["acc", "protein_label"]], on="acc", how="inner")
test_eval = test_eval[(test_eval["split"] == "test") & (test_eval["protein_label"].isin([0, 1]))].copy()

# Single-example demo (prefer a negative case)
neg = test_eval[test_eval["protein_label"] == 0]
demo_row = neg.iloc[0] if len(neg) > 0 else test_eval.iloc[0]
demo_seq = str(demo_row["sequence"])[:SEQ_TRUNC]

x0 = seq_to_original_embedding(demo_seq)
p0 = predict_prob(x0)
print("Demo acc:", demo_row["acc"], "label:", int(demo_row["protein_label"]), "p_before:", round(p0, 6))

for k_lat in K_LATENTS_LIST:
    lat_idx = ranked_latents[:k_lat]
    for a in ALPHAS_TO_TRY:
        x1 = edit_sequence_embedding(demo_seq, lat_idx, alpha=a)
        p1 = predict_prob(x1)
        print({"k_latents": int(k_lat), "alpha": float(a), "p_after": float(p1), "delta_p": float(p1 - p0)})

# Small batch summary
rows = []
subset = test_eval.head(min(80, len(test_eval))).copy()
for k_lat in K_LATENTS_LIST:
    lat_idx = ranked_latents[:k_lat]
    for a in ALPHAS_TO_TRY:
        deltas = []
        for _, r in subset.iterrows():
            seq = str(r["sequence"])[:SEQ_TRUNC]
            p_before = predict_prob(seq_to_original_embedding(seq))
            p_after = predict_prob(edit_sequence_embedding(seq, lat_idx, alpha=a))
            deltas.append(p_after - p_before)
        rows.append({
            "k_latents": int(k_lat),
            "alpha": float(a),
            "n": int(len(deltas)),
            "mean_delta_p": float(np.mean(deltas)) if len(deltas) else np.nan,
            "median_delta_p": float(np.median(deltas)) if len(deltas) else np.nan,
        })

edit_summary_df = pd.DataFrame(rows)
print("\nBatch edit summary:")
display(edit_summary_df)

edit_summary_df.to_csv("edit_summary_disorder.csv", index=False)
print("Saved edit_summary_disorder.csv")

In [ ]:
# Step 6: bidirectional edits + random-latent controls

import numpy as np
import pandas as pd

RANDOM_TRIALS = 5
K_LIST = (1, 5)
ALPHA_LIST = alphas[:3]
N_EVAL = 100

labels_df = pd.read_csv("labels_disorder.csv")
manifest = pd.read_csv("protein_manifest_idpinterp.csv")

test_eval = manifest.merge(labels_df[["acc", "protein_label"]], on="acc", how="inner")
test_eval = test_eval[(test_eval["split"] == "test") & (test_eval["protein_label"].isin([0, 1]))].copy()
subset = test_eval.head(min(N_EVAL, len(test_eval))).copy().reset_index(drop=True)

def run_edit_eval(seqs, latent_indices, alpha):
    deltas = []
    for seq in seqs:
        seq = str(seq)[:SEQ_TRUNC]
        p0 = predict_prob(seq_to_original_embedding(seq))
        p1 = predict_prob(edit_sequence_embedding(seq, latent_indices, alpha=alpha))
        deltas.append(p1 - p0)
    return np.array(deltas, dtype=np.float32)

rows = []
seqs = subset["sequence"].tolist()
d_latent = int(sae.enc.out_features)
rng = np.random.default_rng(SEED)

for k_lat in K_LIST:
    top_idx = ranked_latents[:k_lat]

    for a in ALPHA_LIST:
        for direction in [+1, -1]:
            a_eff = float(direction * a)
            deltas_top = run_edit_eval(seqs, top_idx, alpha=a_eff)
            rows.append({
                "mode": "top",
                "k_latents": int(k_lat),
                "alpha": float(a),
                "direction": int(direction),
                "n": int(len(deltas_top)),
                "mean_delta_p": float(deltas_top.mean()),
                "median_delta_p": float(np.median(deltas_top)),
            })

            # random controls (same k, alpha, direction)
            rand_means, rand_medians = [], []
            for _ in range(RANDOM_TRIALS):
                rand_idx = rng.choice(d_latent, size=k_lat, replace=False)
                deltas_rand = run_edit_eval(seqs, rand_idx, alpha=a_eff)
                rand_means.append(float(deltas_rand.mean()))
                rand_medians.append(float(np.median(deltas_rand)))

            rows.append({
                "mode": "random_avg",
                "k_latents": int(k_lat),
                "alpha": float(a),
                "direction": int(direction),
                "n": int(len(seqs)),
                "mean_delta_p": float(np.mean(rand_means)),
                "median_delta_p": float(np.mean(rand_medians)),
            })

compare_df = pd.DataFrame(rows)

# Convenience view: top vs random side-by-side
pivot_df = compare_df.pivot_table(
    index=["k_latents", "alpha", "direction"],
    columns="mode",
    values=["mean_delta_p", "median_delta_p"],
    aggfunc="first",
)
pivot_df.columns = [f"{m}_{mode}" for (m, mode) in pivot_df.columns]
pivot_df = pivot_df.reset_index().sort_values(["k_latents", "alpha", "direction"])

print("Bidirectional top-vs-random comparison:")
display(pivot_df)

compare_df.to_csv("edit_compare_top_vs_random_disorder.csv", index=False)
pivot_df.to_csv("edit_compare_top_vs_random_disorder_pivot.csv", index=False)
print("Saved edit_compare_top_vs_random_disorder.csv")
print("Saved edit_compare_top_vs_random_disorder_pivot.csv")

In [ ]:
# Step 7: retrieval before/after editing (final loop step)

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Best setting from Step 6 results
BEST_K = 5
BEST_ALPHA = -float(alphas[2])  # direction -1, strongest alpha from your tested grid
TOP_N = 10
N_RETRIEVAL_EVAL = 100

manifest = pd.read_csv("protein_manifest_idpinterp.csv")
labels_df = pd.read_csv("labels_disorder.csv")

meta = manifest[["acc", "split"]].merge(
    labels_df[["acc", "protein_label"]], on="acc", how="inner"
)
meta = meta.merge(embed_df[["acc", "embedding"]], on="acc", how="inner")
meta = meta[meta["protein_label"].isin([0, 1])].copy().reset_index(drop=True)

train_bank = meta[meta["split"] == "train"].copy().reset_index(drop=True)
test_eval = meta[meta["split"] == "test"].copy().reset_index(drop=True)

bank_emb = np.stack(train_bank["embedding"].to_list(), axis=0)
bank_y = train_bank["protein_label"].astype(int).to_numpy()

# Pull sequences from manifest for edits
seq_map = dict(zip(manifest["acc"].astype(str), manifest["sequence"].astype(str)))


def nearest_neighbors(query_emb, bank_emb, top_n=10):
    sims = cosine_similarity(query_emb.reshape(1, -1), bank_emb)[0]
    idx = np.argsort(-sims)[:top_n]
    return idx, sims[idx]

rows = []
lat_idx = ranked_latents[:BEST_K]

for _, r in test_eval.head(min(N_RETRIEVAL_EVAL, len(test_eval))).iterrows():
    acc = str(r["acc"])
    y_true = int(r["protein_label"])
    seq = seq_map[acc][:SEQ_TRUNC]

    x_orig = seq_to_original_embedding(seq)
    x_edit = edit_sequence_embedding(seq, latent_indices=lat_idx, alpha=BEST_ALPHA)

    p_orig = predict_prob(x_orig)
    p_edit = predict_prob(x_edit)

    idx0, sims0 = nearest_neighbors(x_orig, bank_emb, top_n=TOP_N)
    idx1, sims1 = nearest_neighbors(x_edit, bank_emb, top_n=TOP_N)

    pos_rate0 = float(bank_y[idx0].mean())
    pos_rate1 = float(bank_y[idx1].mean())

    rows.append({
        "acc": acc,
        "y_true": y_true,
        "p_orig": p_orig,
        "p_edit": p_edit,
        "delta_p": p_edit - p_orig,
        "orig_neighbor_pos_rate": pos_rate0,
        "edit_neighbor_pos_rate": pos_rate1,
        "delta_neighbor_pos_rate": pos_rate1 - pos_rate0,
        "mean_sim_orig": float(np.mean(sims0)),
        "mean_sim_edit": float(np.mean(sims1)),
    })

retrieval_df = pd.DataFrame(rows)
display(retrieval_df.head())
print("\nRetrieval shift summary:")
display(retrieval_df.describe())
print("mean delta_p:", retrieval_df["delta_p"].mean())
print("mean delta_neighbor_pos_rate:", retrieval_df["delta_neighbor_pos_rate"].mean())

retrieval_df.to_csv("retrieval_shift_disorder.csv", index=False)
print("Saved retrieval_shift_disorder.csv")

In [ ]:
# Step 8A: helper functions for additional properties (kept notebook-local, no config file)

import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


def build_labels_for_term(term, manifest_path="protein_manifest_idpinterp.csv", json_path=JSON_PATH, out_csv=None):
    manifest = pd.read_csv(manifest_path)
    with open(json_path, "r", encoding="utf-8") as f:
        raw_local = json.load(f)

    entries_local = extract_entries(raw_local)

    entry_map_local = {}
    for i, e in enumerate(entries_local):
        acc = get_acc(e, i)
        seq = e.get("sequence")
        if isinstance(seq, str) and seq.strip():
            entry_map_local[acc] = e

    rows = []
    for _, r in manifest.iterrows():
        acc = r["acc"]
        seq = r["sequence"]
        L = len(seq)

        e = entry_map_local.get(acc)
        if e is None:
            rows.append({
                "acc": acc,
                "split": r["split"],
                "length": L,
                "term": term,
                "positive_residues": 0,
                "protein_score": 0.0,
                "protein_label": 0,
            })
            continue

        mask = np.zeros(L, dtype=np.uint8)
        for region in get_regions(e):
            if not isinstance(region, dict):
                continue
            # For non-disorder terms, require term match exactly.
            if term != "disorder":
                if get_region_term(region) != term:
                    continue
            span = get_span(region)
            if span is None:
                continue
            s, t = span
            s = max(1, s)
            t = min(L, t)
            if t >= s:
                mask[s - 1 : t] = 1

        pos = int(mask.sum())
        score = float(pos / L) if L > 0 else 0.0
        # For additional terms, simple binary presence label.
        label = int(pos > 0)

        rows.append({
            "acc": acc,
            "split": r["split"],
            "length": L,
            "term": term,
            "positive_residues": pos,
            "protein_score": score,
            "protein_label": label,
        })

    labels_local = pd.DataFrame(rows)
    if out_csv is None:
        out_csv = f"labels_{term}.csv"
    labels_local.to_csv(out_csv, index=False)

    print(f"Saved {out_csv}")
    print(labels_local.groupby("split")["protein_label"].value_counts(dropna=False))
    return labels_local, out_csv


def train_probe_for_term(term, labels_csv, manifest_path="protein_manifest_idpinterp.csv", embed_cache="embeddings_idpinterp.npz", out_npz=None):
    manifest = pd.read_csv(manifest_path)
    labels_local = pd.read_csv(labels_csv)

    data = np.load(embed_cache, allow_pickle=True)
    X = data["X"]
    accs = data["accs"]
    embed_df_local = pd.DataFrame({"acc": accs.astype(str)})
    embed_df_local["embedding"] = list(X)

    work = manifest[["acc", "split"]].merge(
        labels_local[["acc", "protein_label", "protein_score"]], on="acc", how="inner"
    )
    work = work.merge(embed_df_local, on="acc", how="inner")
    work = work[work["protein_label"].isin([0, 1])].copy()

    X_all = np.stack(work["embedding"].to_list(), axis=0)
    y_all = work["protein_label"].astype(int).to_numpy()

    train_idx = work["split"].values == "train"
    val_idx = work["split"].values == "val"
    test_idx = work["split"].values == "test"

    X_train, y_train = X_all[train_idx], y_all[train_idx]
    X_val, y_val = X_all[val_idx], y_all[val_idx]
    X_test, y_test = X_all[test_idx], y_all[test_idx]

    print(f"[{term}] shapes:", X_train.shape, X_val.shape, X_test.shape)
    print(f"[{term}] train balance:", np.bincount(y_train) if len(y_train) else "empty")

    if len(np.unique(y_train)) < 2:
        print(f"[{term}] skipped: one-class train set")
        return None, None

    clf_local = LogisticRegression(max_iter=3000, class_weight="balanced")
    clf_local.fit(X_train, y_train)

    def _eval(name, Xs, ys):
        p = clf_local.predict_proba(Xs)[:, 1]
        pred = (p >= 0.5).astype(int)
        return {
            "term": term,
            "split": name,
            "acc": float(accuracy_score(ys, pred)),
            "f1": float(f1_score(ys, pred, zero_division=0)),
            "auc": float(roc_auc_score(ys, p)) if len(np.unique(ys)) > 1 else np.nan,
        }

    val_m = _eval("val", X_val, y_val)
    test_m = _eval("test", X_test, y_test)
    print(val_m)
    print(test_m)

    if out_npz is None:
        out_npz = f"probe_data_{term}.npz"
    np.savez(
        out_npz,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test,
    )
    print(f"Saved {out_npz}")

    metrics_df = pd.DataFrame([val_m, test_m])
    metrics_path = f"probe_metrics_{term}.csv"
    metrics_df.to_csv(metrics_path, index=False)
    print(f"Saved {metrics_path}")

    return clf_local, metrics_df

In [ ]:
# Step 8B: property 2/6 - protein_binding

TERM_P2 = "protein_binding"
labels_p2_df, labels_p2_path = build_labels_for_term(TERM_P2)
clf_p2, metrics_p2 = train_probe_for_term(TERM_P2, labels_p2_path)
if metrics_p2 is not None:
    display(metrics_p2)

In [ ]:
# Step 8C: property 3/6 - disorder_to_order

TERM_P3 = "disorder_to_order"
labels_p3_df, labels_p3_path = build_labels_for_term(TERM_P3)
clf_p3, metrics_p3 = train_probe_for_term(TERM_P3, labels_p3_path)
if metrics_p3 is not None:
    display(metrics_p3)

In [ ]:
# Step 8D: property 4/6 - molecular_adaptor_activity

TERM_P4 = "molecular_adaptor_activity"
labels_p4_df, labels_p4_path = build_labels_for_term(TERM_P4)
clf_p4, metrics_p4 = train_probe_for_term(TERM_P4, labels_p4_path)
if metrics_p4 is not None:
    display(metrics_p4)

In [ ]:
# Step 8E: property 5/6 - flexible_linker

TERM_P5 = "flexible_linker"
labels_p5_df, labels_p5_path = build_labels_for_term(TERM_P5)
clf_p5, metrics_p5 = train_probe_for_term(TERM_P5, labels_p5_path)
if metrics_p5 is not None:
    display(metrics_p5)

In [ ]:
# Step 8F: property 6/6 - pre_molten_globule

TERM_P6 = "pre_molten_globule"
labels_p6_df, labels_p6_path = build_labels_for_term(TERM_P6)
clf_p6, metrics_p6 = train_probe_for_term(TERM_P6, labels_p6_path)
if metrics_p6 is not None:
    display(metrics_p6)

In [ ]:
# Step 8G: summarize probe metrics across the 5 additional properties

import glob
import pandas as pd

metric_files = sorted(glob.glob("probe_metrics_*.csv"))
if len(metric_files) == 0:
    print("No probe metric files found yet. Run Step 8B-8F first.")
else:
    all_metrics = pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True)
    all_metrics = all_metrics.sort_values(["term", "split"]).reset_index(drop=True)
    display(all_metrics)

    pivot = all_metrics.pivot(index="term", columns="split", values=["auc", "f1", "acc"])
    pivot.columns = [f"{m}_{s}" for (m, s) in pivot.columns]
    pivot = pivot.reset_index().sort_values("term")
    print("\nPivot summary:")
    display(pivot)

    pivot.to_csv("probe_metrics_all_terms_pivot.csv", index=False)
    print("Saved probe_metrics_all_terms_pivot.csv")

In [ ]:
# Step 9A: helpers to rescue weak/imbalanced terms

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score


def build_strict_labels_from_scores(term, labels_csv, pos_quantile_nonzero=0.7):
    """
    Build stricter binary labels from protein_score:
      - negative: score == 0
      - positive: score >= quantile(score>0)
      - middle nonzero scores dropped (label=-1)
    """
    df = pd.read_csv(labels_csv).copy()
    df = df[df["term"] == term].copy()

    nonzero = df[df["protein_score"] > 0]["protein_score"]
    if len(nonzero) == 0:
        raise ValueError(f"[{term}] no nonzero protein_score found")

    cutoff = float(nonzero.quantile(pos_quantile_nonzero))
    df["protein_label"] = -1
    df.loc[df["protein_score"] == 0, "protein_label"] = 0
    df.loc[df["protein_score"] >= cutoff, "protein_label"] = 1

    print(f"[{term}] strict cutoff@q={pos_quantile_nonzero}: {cutoff:.6f}")
    print(df.groupby("split")["protein_label"].value_counts(dropna=False))
    return df, cutoff


def train_with_threshold_tuning(term, labels_df, embed_cache="embeddings_idpinterp.npz", manifest_path="protein_manifest_idpinterp.csv"):
    """
    Train logistic probe on labels in labels_df and tune threshold on validation F1.
    labels_df must contain: acc, split, protein_label, protein_score
    """
    manifest = pd.read_csv(manifest_path)
    emb = np.load(embed_cache, allow_pickle=True)
    embed_df_local = pd.DataFrame({"acc": emb["accs"].astype(str)})
    embed_df_local["embedding"] = list(emb["X"])

    work = manifest[["acc", "split"]].merge(
        labels_df[["acc", "protein_label", "protein_score"]], on="acc", how="inner"
    )
    work = work.merge(embed_df_local, on="acc", how="inner")
    work = work[work["protein_label"].isin([0, 1])].copy()

    X_all = np.stack(work["embedding"].to_list(), axis=0)
    y_all = work["protein_label"].astype(int).to_numpy()

    train_idx = work["split"].values == "train"
    val_idx = work["split"].values == "val"
    test_idx = work["split"].values == "test"

    X_train, y_train = X_all[train_idx], y_all[train_idx]
    X_val, y_val = X_all[val_idx], y_all[val_idx]
    X_test, y_test = X_all[test_idx], y_all[test_idx]

    print(f"[{term}] shapes:", X_train.shape, X_val.shape, X_test.shape)
    if len(y_train) == 0 or len(np.unique(y_train)) < 2:
        raise ValueError(f"[{term}] train split is one-class or empty")

    clf_local = LogisticRegression(max_iter=5000, class_weight="balanced")
    clf_local.fit(X_train, y_train)

    p_val = clf_local.predict_proba(X_val)[:, 1]
    p_test = clf_local.predict_proba(X_test)[:, 1]

    # Tune threshold on val F1
    thr_grid = np.linspace(0.05, 0.95, 37)
    best_thr, best_f1 = 0.5, -1.0
    for t in thr_grid:
        f1 = f1_score(y_val, (p_val >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = float(f1)
            best_thr = float(t)

    def _metrics(split, y, p, thr):
        pred = (p >= thr).astype(int)
        return {
            "term": term,
            "split": split,
            "threshold": float(thr),
            "acc": float(accuracy_score(y, pred)),
            "f1": float(f1_score(y, pred, zero_division=0)),
            "auc": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
            "auprc": float(average_precision_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
            "n": int(len(y)),
            "pos_rate": float(y.mean()) if len(y) else np.nan,
        }

    val_m = _metrics("val", y_val, p_val, best_thr)
    test_m = _metrics("test", y_test, p_test, best_thr)
    print(f"[{term}] best val threshold: {best_thr:.3f}")
    print(val_m)
    print(test_m)

    out = pd.DataFrame([val_m, test_m])
    return clf_local, out

In [ ]:
# Step 9B: rescue attempt for molecular_adaptor_activity

TERM_FIX_1 = "molecular_adaptor_activity"
base_labels_fix1 = pd.read_csv(f"labels_{TERM_FIX_1}.csv")

# Try a few strictness levels; keep best by validation AUC
candidates = [0.50, 0.70, 0.85]
rows = []
best_df = None
best_score = -np.inf
best_q = None

for q in candidates:
    strict_df, cutoff = build_strict_labels_from_scores(
        TERM_FIX_1,
        labels_csv=f"labels_{TERM_FIX_1}.csv",
        pos_quantile_nonzero=q,
    )
    _, metrics_df = train_with_threshold_tuning(TERM_FIX_1, strict_df)
    val_auc = float(metrics_df[metrics_df["split"] == "val"]["auc"].iloc[0])

    rows.append({
        "term": TERM_FIX_1,
        "q_nonzero": q,
        "cutoff": cutoff,
        "val_auc": val_auc,
        "val_f1": float(metrics_df[metrics_df["split"] == "val"]["f1"].iloc[0]),
        "test_auc": float(metrics_df[metrics_df["split"] == "test"]["auc"].iloc[0]),
        "test_f1": float(metrics_df[metrics_df["split"] == "test"]["f1"].iloc[0]),
    })

    if val_auc > best_score:
        best_score = val_auc
        best_df = metrics_df.copy()
        best_q = q

rescue_scan_fix1 = pd.DataFrame(rows).sort_values("val_auc", ascending=False)
print("\nMolecular adaptor rescue scan:")
display(rescue_scan_fix1)
print(f"Best q_nonzero by val AUC: {best_q}")
display(best_df)

rescue_scan_fix1.to_csv("rescue_scan_molecular_adaptor_activity.csv", index=False)
best_df.to_csv("rescue_best_metrics_molecular_adaptor_activity.csv", index=False)
print("Saved rescue scan + best metrics for molecular_adaptor_activity")

In [ ]:
# Step 9C: rescue attempt for pre_molten_globule (extreme imbalance)

TERM_FIX_2 = "pre_molten_globule"
labels_fix2 = pd.read_csv(f"labels_{TERM_FIX_2}.csv")

# Keep binary labels but tune decision threshold on val + report AUPRC.
_, metrics_fix2 = train_with_threshold_tuning(TERM_FIX_2, labels_fix2)
print("\nPre-molten-globule tuned metrics:")
display(metrics_fix2)

metrics_fix2.to_csv("rescue_best_metrics_pre_molten_globule.csv", index=False)
print("Saved rescue_best_metrics_pre_molten_globule.csv")

In [ ]:
# Step 9D: compare original vs rescued metrics for the two problematic terms

orig_all = pd.read_csv("probe_metrics_all_terms_pivot.csv")

orig_subset = orig_all[orig_all["term"].isin(["molecular_adaptor_activity", "pre_molten_globule"])].copy()
orig_subset["version"] = "original"
orig_subset = orig_subset[["term", "version", "auc_val", "auc_test", "f1_val", "f1_test", "acc_val", "acc_test"]]

resc_parts = []

if os.path.exists("rescue_best_metrics_molecular_adaptor_activity.csv"):
    r1 = pd.read_csv("rescue_best_metrics_molecular_adaptor_activity.csv")
    r1p = r1.pivot(index="term", columns="split", values=["auc", "f1", "acc"])
    r1p.columns = [f"{m}_{s}" for (m, s) in r1p.columns]
    r1p = r1p.reset_index()
    r1p["version"] = "rescued"
    resc_parts.append(r1p[["term", "version", "auc_val", "auc_test", "f1_val", "f1_test", "acc_val", "acc_test"]])

if os.path.exists("rescue_best_metrics_pre_molten_globule.csv"):
    r2 = pd.read_csv("rescue_best_metrics_pre_molten_globule.csv")
    r2p = r2.pivot(index="term", columns="split", values=["auc", "f1", "acc"])
    r2p.columns = [f"{m}_{s}" for (m, s) in r2p.columns]
    r2p = r2p.reset_index()
    r2p["version"] = "rescued"
    resc_parts.append(r2p[["term", "version", "auc_val", "auc_test", "f1_val", "f1_test", "acc_val", "acc_test"]])

if len(resc_parts) > 0:
    resc_subset = pd.concat(resc_parts, ignore_index=True)
    comp = pd.concat([orig_subset, resc_subset], ignore_index=True).sort_values(["term", "version"])
    display(comp)
    comp.to_csv("rescue_compare_problem_terms.csv", index=False)
    print("Saved rescue_compare_problem_terms.csv")
else:
    print("No rescue metric files found. Run Step 9B and 9C first.")

In [ ]:
# Step 10: auto-build final term summary leaderboard

import os
import glob
import numpy as np
import pandas as pd

TERMS_KEEP = [
    "disorder",
    "protein_binding",
    "disorder_to_order",
    "flexible_linker",
]

rows = []
for term in TERMS_KEEP:
    row = {"term": term}

    # -----------------------------
    # Probe metrics
    # -----------------------------
    probe_path = f"probe_metrics_{term}.csv"
    if os.path.exists(probe_path):
        pm = pd.read_csv(probe_path)
        val = pm[pm["split"] == "val"]
        test = pm[pm["split"] == "test"]
        row["probe_auc_val"] = float(val["auc"].iloc[0]) if len(val) else np.nan
        row["probe_auc_test"] = float(test["auc"].iloc[0]) if len(test) else np.nan
        row["probe_f1_val"] = float(val["f1"].iloc[0]) if len(val) else np.nan
        row["probe_f1_test"] = float(test["f1"].iloc[0]) if len(test) else np.nan
        row["probe_acc_val"] = float(val["acc"].iloc[0]) if len(val) else np.nan
        row["probe_acc_test"] = float(test["acc"].iloc[0]) if len(test) else np.nan
    else:
        row.update({
            "probe_auc_val": np.nan,
            "probe_auc_test": np.nan,
            "probe_f1_val": np.nan,
            "probe_f1_test": np.nan,
            "probe_acc_val": np.nan,
            "probe_acc_test": np.nan,
        })

    # -----------------------------
    # Edit comparison (top vs random)
    # -----------------------------
    edit_pivot_candidates = [
        f"edit_compare_top_vs_random_{term}_pivot.csv",
        f"edit_compare_top_vs_random_{term}.csv",  # fallback long format
    ]

    edit_loaded = False
    for ep in edit_pivot_candidates:
        if not os.path.exists(ep):
            continue

        ed = pd.read_csv(ep)
        # If long format, pivot into expected columns.
        if "mode" in ed.columns:
            pv = ed.pivot_table(
                index=["k_latents", "alpha", "direction"],
                columns="mode",
                values=["mean_delta_p", "median_delta_p"],
                aggfunc="first",
            )
            pv.columns = [f"{m}_{mode}" for (m, mode) in pv.columns]
            ed = pv.reset_index()

        needed = {"mean_delta_p_top", "mean_delta_p_random_avg"}
        if not needed.issubset(set(ed.columns)):
            continue

        ed = ed.copy()
        ed["top_minus_random"] = ed["mean_delta_p_top"] - ed["mean_delta_p_random_avg"]
        best_i = ed["top_minus_random"].idxmax()
        best = ed.loc[best_i]

        row["best_k_latents"] = int(best["k_latents"])
        row["best_alpha"] = float(best["alpha"])
        row["best_direction"] = int(best["direction"])
        row["edit_mean_delta_p_top"] = float(best["mean_delta_p_top"])
        row["edit_mean_delta_p_random"] = float(best["mean_delta_p_random_avg"])
        row["edit_top_minus_random"] = float(best["top_minus_random"])

        # Optional median columns if present
        row["edit_median_delta_p_top"] = float(best["median_delta_p_top"]) if "median_delta_p_top" in best else np.nan
        row["edit_median_delta_p_random"] = float(best["median_delta_p_random_avg"]) if "median_delta_p_random_avg" in best else np.nan

        edit_loaded = True
        break

    if not edit_loaded:
        row.update({
            "best_k_latents": np.nan,
            "best_alpha": np.nan,
            "best_direction": np.nan,
            "edit_mean_delta_p_top": np.nan,
            "edit_mean_delta_p_random": np.nan,
            "edit_top_minus_random": np.nan,
            "edit_median_delta_p_top": np.nan,
            "edit_median_delta_p_random": np.nan,
        })

    # -----------------------------
    # Retrieval shift
    # -----------------------------
    retrieval_path = f"retrieval_shift_{term}.csv"
    if os.path.exists(retrieval_path):
        rd = pd.read_csv(retrieval_path)
        row["retrieval_mean_delta_p"] = float(rd["delta_p"].mean()) if "delta_p" in rd.columns else np.nan
        row["retrieval_mean_delta_neighbor_pos_rate"] = (
            float(rd["delta_neighbor_pos_rate"].mean()) if "delta_neighbor_pos_rate" in rd.columns else np.nan
        )
    else:
        row["retrieval_mean_delta_p"] = np.nan
        row["retrieval_mean_delta_neighbor_pos_rate"] = np.nan

    rows.append(row)

final_summary = pd.DataFrame(rows)

# Handy ranking score for quick decision-making.
# (Probe quality + intervention advantage + retrieval shift)
final_summary["priority_score"] = (
    final_summary["probe_auc_test"].fillna(0) * 0.5
    + final_summary["edit_top_minus_random"].fillna(0) * 20.0
    + final_summary["retrieval_mean_delta_neighbor_pos_rate"].fillna(0) * 10.0
)

final_summary = final_summary.sort_values("priority_score", ascending=False).reset_index(drop=True)

display(final_summary)
final_summary.to_csv("final_term_summary.csv", index=False)
print("Saved final_term_summary.csv")

# Also save a compact leaderboard.
leaderboard_cols = [
    "term",
    "probe_auc_test",
    "probe_f1_test",
    "edit_top_minus_random",
    "retrieval_mean_delta_neighbor_pos_rate",
    "priority_score",
]
leaderboard = final_summary[leaderboard_cols].copy()
display(leaderboard)
leaderboard.to_csv("final_term_leaderboard.csv", index=False)
print("Saved final_term_leaderboard.csv")

In [ ]:
# Step 10A: ensure disorder probe metrics file exists (fallback)

import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

if not os.path.exists("probe_metrics_disorder.csv"):
    if os.path.exists("probe_disorder_data.npz"):
        d = np.load("probe_disorder_data.npz")
        X_train, y_train = d["X_train"], d["y_train"]
        X_val, y_val = d["X_val"], d["y_val"]
        X_test, y_test = d["X_test"], d["y_test"]

        clf_tmp = LogisticRegression(max_iter=3000, class_weight="balanced")
        clf_tmp.fit(X_train, y_train)

        def _m(split, Xs, ys):
            p = clf_tmp.predict_proba(Xs)[:, 1]
            pred = (p >= 0.5).astype(int)
            return {
                "term": "disorder",
                "split": split,
                "acc": float(accuracy_score(ys, pred)),
                "f1": float(f1_score(ys, pred, zero_division=0)),
                "auc": float(roc_auc_score(ys, p)) if len(np.unique(ys)) > 1 else np.nan,
            }

        metrics_dis = pd.DataFrame([_m("val", X_val, y_val), _m("test", X_test, y_test)])
        metrics_dis.to_csv("probe_metrics_disorder.csv", index=False)
        print("Saved probe_metrics_disorder.csv from probe_disorder_data.npz")
        display(metrics_dis)
    else:
        print("probe_disorder_data.npz not found; cannot build probe_metrics_disorder.csv fallback")
else:
    print("probe_metrics_disorder.csv already exists")

In [ ]:
# Step 11A: full non-disorder loop helper (probe -> latent rank -> edit compare -> retrieval)

import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


def run_full_term_loop(term, labels_csv, n_rank=800, n_edit_eval=100, random_trials=5, top_n_retrieval=10):
    """Runs full loop for one term and saves term-specific artifacts."""

    # -------------------------
    # 1) Probe
    # -------------------------
    clf_local, metrics_df = train_probe_for_term(term, labels_csv)
    if clf_local is None or metrics_df is None:
        print(f"[{term}] skipped: probe not trainable")
        return None

    # -------------------------
    # 2) Latent ranking
    # -------------------------
    labels_df = pd.read_csv(labels_csv)
    manifest = pd.read_csv("protein_manifest_idpinterp.csv")

    train_labeled = manifest.merge(labels_df[["acc", "protein_label"]], on="acc", how="inner")
    train_labeled = train_labeled[
        (train_labeled["split"] == "train") & (train_labeled["protein_label"].isin([0, 1]))
    ].copy().reset_index(drop=True)

    if len(train_labeled) == 0:
        print(f"[{term}] no train rows after binary filtering")
        return None

    @torch.no_grad()
    def _protein_latent_summary(seq, reduce="max"):
        h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
        h = h[:K_TOKENS]
        if h.shape[0] == 0:
            return np.zeros(sae.enc.out_features, dtype=np.float32)
        z = torch.relu(sae.enc(h))
        if reduce == "max":
            Z = z.max(dim=0).values
        else:
            Z = z.mean(dim=0)
        return Z.detach().cpu().numpy().astype(np.float32)

    def _rank_latents(Zmat, y):
        Zc = (Zmat - Zmat.mean(axis=0)) / (Zmat.std(axis=0) + 1e-8)
        yc = (y - y.mean()) / (y.std() + 1e-8)
        corr = (Zc * yc[:, None]).mean(axis=0)
        ranked = np.argsort(-np.abs(corr))
        return ranked, corr

    rank_cache = f"latent_rank_{term}.npz"
    if os.path.exists(rank_cache):
        rc = np.load(rank_cache)
        ranked_latents = rc["ranked_latents"]
        corr = rc["corr"]
        alphas_local = rc["alphas"]
    else:
        M = min(n_rank, len(train_labeled))
        seqsM = [s[:SEQ_TRUNC] for s in train_labeled["sequence"].tolist()[:M]]
        yM = train_labeled["protein_label"].to_numpy(dtype=np.float32)[:M]
        Zmat = np.stack([_protein_latent_summary(seq) for seq in seqsM], axis=0)
        ranked_latents, corr = _rank_latents(Zmat, yM)
        top1_sd = Zmat[:, int(ranked_latents[0])].std() + 1e-8
        alphas_local = np.array([0, 1*top1_sd, 2*top1_sd, 4*top1_sd], dtype=np.float32)
        np.savez(rank_cache, ranked_latents=ranked_latents, corr=corr, alphas=alphas_local)

    # -------------------------
    # 3) Edit compare top vs random (bidirectional)
    # -------------------------
    emb = np.load("embeddings_idpinterp.npz", allow_pickle=True)
    embed_df_local = pd.DataFrame({"acc": emb["accs"].astype(str)})
    embed_df_local["embedding"] = list(emb["X"])

    test_eval = manifest.merge(labels_df[["acc", "protein_label"]], on="acc", how="inner")
    test_eval = test_eval[(test_eval["split"] == "test") & (test_eval["protein_label"].isin([0, 1]))].copy()
    test_eval = test_eval.head(min(n_edit_eval, len(test_eval))).copy().reset_index(drop=True)

    @torch.no_grad()
    def _seq_to_orig_emb(seq):
        h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
        h = h[:K_TOKENS]
        if h.shape[0] == 0:
            return np.zeros(480, dtype=np.float32)
        return h.mean(dim=0).detach().cpu().numpy().astype(np.float32)

    @torch.no_grad()
    def _edit_emb(seq, latent_indices, alpha, edit_first_k=30):
        h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
        h = h[:K_TOKENS]
        if h.shape[0] == 0:
            return _seq_to_orig_emb(seq)
        z = torch.relu(sae.enc(h))
        z_edit = z.clone()
        k = min(edit_first_k, z_edit.shape[0])
        z_edit[:k, latent_indices] += float(alpha)
        h_hat = sae.dec(z_edit)
        return h_hat.mean(dim=0).detach().cpu().numpy().astype(np.float32)

    def _predict_prob(x):
        return float(clf_local.predict_proba(x.reshape(1, -1))[0, 1])

    def _run_eval(seqs, latent_indices, alpha):
        deltas = []
        for s in seqs:
            s = str(s)[:SEQ_TRUNC]
            p0 = _predict_prob(_seq_to_orig_emb(s))
            p1 = _predict_prob(_edit_emb(s, latent_indices, alpha))
            deltas.append(p1 - p0)
        return np.array(deltas, dtype=np.float32)

    rows_cmp = []
    seqs = test_eval["sequence"].tolist()
    d_latent = int(sae.enc.out_features)
    rng = np.random.default_rng(SEED)

    for k_lat in (1, 5):
        top_idx = ranked_latents[:k_lat]
        for a in alphas_local[:3]:
            for direction in (+1, -1):
                a_eff = float(direction * a)
                d_top = _run_eval(seqs, top_idx, a_eff)
                rows_cmp.append({
                    "mode": "top",
                    "k_latents": int(k_lat),
                    "alpha": float(a),
                    "direction": int(direction),
                    "mean_delta_p": float(d_top.mean()),
                    "median_delta_p": float(np.median(d_top)),
                })

                rand_means, rand_medians = [], []
                for _ in range(random_trials):
                    r_idx = rng.choice(d_latent, size=k_lat, replace=False)
                    d_rand = _run_eval(seqs, r_idx, a_eff)
                    rand_means.append(float(d_rand.mean()))
                    rand_medians.append(float(np.median(d_rand)))

                rows_cmp.append({
                    "mode": "random_avg",
                    "k_latents": int(k_lat),
                    "alpha": float(a),
                    "direction": int(direction),
                    "mean_delta_p": float(np.mean(rand_means)),
                    "median_delta_p": float(np.mean(rand_medians)),
                })

    cmp_df = pd.DataFrame(rows_cmp)
    cmp_df.to_csv(f"edit_compare_top_vs_random_{term}.csv", index=False)

    pv = cmp_df.pivot_table(
        index=["k_latents", "alpha", "direction"],
        columns="mode",
        values=["mean_delta_p", "median_delta_p"],
        aggfunc="first",
    )
    pv.columns = [f"{m}_{mode}" for (m, mode) in pv.columns]
    pv = pv.reset_index()
    pv["top_minus_random"] = pv["mean_delta_p_top"] - pv["mean_delta_p_random_avg"]
    pv.to_csv(f"edit_compare_top_vs_random_{term}_pivot.csv", index=False)

    best = pv.loc[pv["top_minus_random"].idxmax()]
    best_k = int(best["k_latents"])
    best_alpha = float(best["alpha"])
    best_direction = int(best["direction"])
    best_alpha_eff = best_alpha * best_direction

    # -------------------------
    # 4) Retrieval before/after with best setting
    # -------------------------
    meta = manifest[["acc", "split"]].merge(labels_df[["acc", "protein_label"]], on="acc", how="inner")
    meta = meta.merge(embed_df_local[["acc", "embedding"]], on="acc", how="inner")
    meta = meta[meta["protein_label"].isin([0, 1])].copy().reset_index(drop=True)

    train_bank = meta[meta["split"] == "train"].copy().reset_index(drop=True)
    test_bank = meta[meta["split"] == "test"].copy().reset_index(drop=True)

    bank_emb = np.stack(train_bank["embedding"].to_list(), axis=0)
    bank_y = train_bank["protein_label"].astype(int).to_numpy()

    seq_map = dict(zip(manifest["acc"].astype(str), manifest["sequence"].astype(str)))

    def _nn(query, top_n=10):
        sims = cosine_similarity(query.reshape(1, -1), bank_emb)[0]
        idx = np.argsort(-sims)[:top_n]
        return idx, sims[idx]

    rows_ret = []
    lat_idx_best = ranked_latents[:best_k]
    for _, r in test_bank.head(min(100, len(test_bank))).iterrows():
        acc = str(r["acc"])
        seq = seq_map[acc][:SEQ_TRUNC]
        y_true = int(r["protein_label"])

        x0 = _seq_to_orig_emb(seq)
        x1 = _edit_emb(seq, lat_idx_best, best_alpha_eff)

        p0 = _predict_prob(x0)
        p1 = _predict_prob(x1)

        i0, s0 = _nn(x0, top_n=top_n_retrieval)
        i1, s1 = _nn(x1, top_n=top_n_retrieval)

        rows_ret.append({
            "acc": acc,
            "y_true": y_true,
            "p_orig": p0,
            "p_edit": p1,
            "delta_p": p1 - p0,
            "orig_neighbor_pos_rate": float(bank_y[i0].mean()),
            "edit_neighbor_pos_rate": float(bank_y[i1].mean()),
            "delta_neighbor_pos_rate": float(bank_y[i1].mean() - bank_y[i0].mean()),
            "mean_sim_orig": float(np.mean(s0)),
            "mean_sim_edit": float(np.mean(s1)),
        })

    ret_df = pd.DataFrame(rows_ret)
    ret_df.to_csv(f"retrieval_shift_{term}.csv", index=False)

    print(f"[{term}] done. best edit: k={best_k}, alpha={best_alpha:.4f}, direction={best_direction}")
    print(f"[{term}] probe test AUC={float(metrics_df[metrics_df['split']=='test']['auc'].iloc[0]):.4f}")
    print(f"[{term}] mean top-random delta={float(best['top_minus_random']):.6f}")
    print(f"[{term}] mean retrieval delta neighbor pos rate={ret_df['delta_neighbor_pos_rate'].mean():.6f}")

    return {
        "term": term,
        "probe_metrics": metrics_df,
        "edit_pivot": pv,
        "retrieval": ret_df,
    }

In [ ]:
# Step 11B: run full loop for protein_binding

res_protein_binding = run_full_term_loop(
    term="protein_binding",
    labels_csv="labels_protein_binding.csv",
)
print("protein_binding loop complete")

In [ ]:
# Step 11C: run full loop for disorder_to_order

res_disorder_to_order = run_full_term_loop(
    term="disorder_to_order",
    labels_csv="labels_disorder_to_order.csv",
)
print("disorder_to_order loop complete")

In [ ]:
# Step 11D: run full loop for flexible_linker

res_flexible_linker = run_full_term_loop(
    term="flexible_linker",
    labels_csv="labels_flexible_linker.csv",
)
print("flexible_linker loop complete")

In [ ]:
# Step 11E: rebuild final summary after full loops

# Re-run Step 10 cell after Step 11B/11C/11D so all terms have probe+edit+retrieval fields.
print("Now run Step 10 again to refresh final_term_summary.csv and final_term_leaderboard.csv")

In [ ]:
# Step 12A: primary-claim setup (disorder + disorder_to_order), frozen seeds/settings, bootstrap utils

import json
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(SEED)

PRIMARY_TERMS = ["disorder", "disorder_to_order"]
PRIMARY_LABEL_PATHS = {
    "disorder": "labels_disorder.csv",
    "disorder_to_order": "labels_disorder_to_order.csv",
}

# Search grid for stronger retrieval effects
SEARCH_GRID = {
    "k_latents": [1, 5, 10],
    "alpha_mult": [1.0, 2.0, 4.0],
    "direction": [-1, 1],
    "edit_first_k": [15, 30, 60],
}

PRIMARY_CONFIG = {
    "seed": int(SEED),
    "terms": PRIMARY_TERMS,
    "search_grid": SEARCH_GRID,
    "near_boundary_n": 120,
    "top_n_retrieval": 10,
    "random_trials": 8,
    "bootstrap_n": 1000,
}

with open("primary_terms_config.json", "w") as f:
    json.dump(PRIMARY_CONFIG, f, indent=2)
print("Saved primary_terms_config.json")


def bootstrap_ci(values, n_boot=1000, seed=42, alpha=0.05):
    vals = np.asarray(values, dtype=np.float32)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(n_boot):
        sample = rng.choice(vals, size=len(vals), replace=True)
        means.append(float(np.mean(sample)))
    means = np.array(means, dtype=np.float32)
    lo = float(np.quantile(means, alpha / 2))
    hi = float(np.quantile(means, 1 - alpha / 2))
    return float(np.mean(vals)), lo, hi


def load_rank_info_for_term(term):
    rank_path = f"latent_rank_{term}.npz"
    if not os.path.exists(rank_path):
        raise FileNotFoundError(f"Missing {rank_path}. Run full loop/ranking for {term} first.")
    z = np.load(rank_path)
    return z["ranked_latents"], z["alphas"]

print("Primary setup ready")

In [ ]:
# Step 12B: build reusable term contexts + near-boundary subset selector

manifest = pd.read_csv("protein_manifest_idpinterp.csv")
emb = np.load("embeddings_idpinterp.npz", allow_pickle=True)
embed_df_global = pd.DataFrame({"acc": emb["accs"].astype(str)})
embed_df_global["embedding"] = list(emb["X"])


def build_term_context(term, labels_csv):
    labels_df = pd.read_csv(labels_csv)

    # train probe once for this term
    clf_term, metrics_term = train_probe_for_term(term, labels_csv)
    if clf_term is None:
        raise ValueError(f"[{term}] probe is not trainable")

    data = manifest[["acc", "split", "sequence"]].merge(
        labels_df[["acc", "protein_label", "protein_score"]], on="acc", how="inner"
    ).merge(embed_df_global[["acc", "embedding"]], on="acc", how="inner")
    data = data[data["protein_label"].isin([0, 1])].copy().reset_index(drop=True)

    train_bank = data[data["split"] == "train"].copy().reset_index(drop=True)
    test_set = data[data["split"] == "test"].copy().reset_index(drop=True)

    bank_emb = np.stack(train_bank["embedding"].to_list(), axis=0)
    bank_y = train_bank["protein_label"].astype(int).to_numpy()

    ranked_latents, alphas_local = load_rank_info_for_term(term)

    # near-boundary subset: smallest absolute classifier margins
    X_test = np.stack(test_set["embedding"].to_list(), axis=0)
    margins = np.abs(clf_term.decision_function(X_test))
    near_idx = np.argsort(margins)[:min(PRIMARY_CONFIG["near_boundary_n"], len(test_set))]
    test_near = test_set.iloc[near_idx].copy().reset_index(drop=True)

    return {
        "term": term,
        "clf": clf_term,
        "metrics": metrics_term,
        "data": data,
        "train_bank": train_bank,
        "test_set": test_set,
        "test_near": test_near,
        "bank_emb": bank_emb,
        "bank_y": bank_y,
        "ranked_latents": ranked_latents,
        "alphas": alphas_local,
    }


def seq_to_orig_emb_term(seq):
    h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
    h = h[:K_TOKENS]
    if h.shape[0] == 0:
        return np.zeros(480, dtype=np.float32)
    return h.mean(dim=0).detach().cpu().numpy().astype(np.float32)


def seq_to_edit_emb_term(seq, latent_indices, alpha, edit_first_k):
    h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
    h = h[:K_TOKENS]
    if h.shape[0] == 0:
        return seq_to_orig_emb_term(seq)

    z = torch.relu(sae.enc(h))
    z_edit = z.clone()
    k = min(int(edit_first_k), z_edit.shape[0])
    z_edit[:k, latent_indices] += float(alpha)
    h_hat = sae.dec(z_edit)
    return h_hat.mean(dim=0).detach().cpu().numpy().astype(np.float32)


primary_contexts = {}
for t in PRIMARY_TERMS:
    primary_contexts[t] = build_term_context(t, PRIMARY_LABEL_PATHS[t])
    print(f"[{t}] context ready: test={len(primary_contexts[t]['test_set'])}, near={len(primary_contexts[t]['test_near'])}")

In [ ]:
# Step 12C: tune retrieval effects on near-boundary examples + strong controls + bootstrap CIs


def evaluate_setting(ctx, k_latents, alpha, direction, edit_first_k, random_trials=8, top_n=10):
    term = ctx["term"]
    clf_term = ctx["clf"]
    test_df = ctx["test_near"]
    bank_emb = ctx["bank_emb"]
    bank_y = ctx["bank_y"]
    lat_idx = ctx["ranked_latents"][:int(k_latents)]
    alpha_eff = float(alpha) * int(direction)

    def _prob(x):
        return float(clf_term.predict_proba(x.reshape(1, -1))[0, 1])

    def _nn(query):
        sims = cosine_similarity(query.reshape(1, -1), bank_emb)[0]
        idx = np.argsort(-sims)[:top_n]
        return idx

    delta_p_top, delta_retr_top = [], []
    seqs = test_df["sequence"].tolist()

    # top-latent edits
    for seq in seqs:
        seq = str(seq)[:SEQ_TRUNC]
        x0 = seq_to_orig_emb_term(seq)
        x1 = seq_to_edit_emb_term(seq, lat_idx, alpha_eff, edit_first_k)
        p0, p1 = _prob(x0), _prob(x1)
        i0, i1 = _nn(x0), _nn(x1)
        delta_p_top.append(p1 - p0)
        delta_retr_top.append(float(bank_y[i1].mean() - bank_y[i0].mean()))

    # random-latent controls
    rng = np.random.default_rng(SEED)
    d_lat = int(sae.enc.out_features)
    rand_delta_p_means, rand_delta_r_means = [], []
    for _ in range(random_trials):
        r_idx = rng.choice(d_lat, size=int(k_latents), replace=False)
        dp, dr = [], []
        for seq in seqs:
            seq = str(seq)[:SEQ_TRUNC]
            x0 = seq_to_orig_emb_term(seq)
            x1 = seq_to_edit_emb_term(seq, r_idx, alpha_eff, edit_first_k)
            p0, p1 = _prob(x0), _prob(x1)
            i0, i1 = _nn(x0), _nn(x1)
            dp.append(p1 - p0)
            dr.append(float(bank_y[i1].mean() - bank_y[i0].mean()))
        rand_delta_p_means.append(float(np.mean(dp)))
        rand_delta_r_means.append(float(np.mean(dr)))

    # sign-flip control (same top latents, opposite direction)
    sf_delta_p, sf_delta_r = [], []
    alpha_sf = -alpha_eff
    for seq in seqs:
        seq = str(seq)[:SEQ_TRUNC]
        x0 = seq_to_orig_emb_term(seq)
        x1 = seq_to_edit_emb_term(seq, lat_idx, alpha_sf, edit_first_k)
        p0, p1 = _prob(x0), _prob(x1)
        i0, i1 = _nn(x0), _nn(x1)
        sf_delta_p.append(p1 - p0)
        sf_delta_r.append(float(bank_y[i1].mean() - bank_y[i0].mean()))

    # bootstrap CIs for top edit effects
    mean_dp, dp_lo, dp_hi = bootstrap_ci(delta_p_top, n_boot=PRIMARY_CONFIG["bootstrap_n"], seed=SEED)
    mean_dr, dr_lo, dr_hi = bootstrap_ci(delta_retr_top, n_boot=PRIMARY_CONFIG["bootstrap_n"], seed=SEED)

    return {
        "term": term,
        "k_latents": int(k_latents),
        "alpha": float(alpha),
        "direction": int(direction),
        "edit_first_k": int(edit_first_k),
        "n_eval": int(len(seqs)),
        "mean_delta_p_top": float(np.mean(delta_p_top)),
        "mean_delta_retr_top": float(np.mean(delta_retr_top)),
        "mean_delta_p_random": float(np.mean(rand_delta_p_means)),
        "mean_delta_retr_random": float(np.mean(rand_delta_r_means)),
        "mean_delta_p_signflip": float(np.mean(sf_delta_p)),
        "mean_delta_retr_signflip": float(np.mean(sf_delta_r)),
        "top_minus_random_delta_p": float(np.mean(delta_p_top) - np.mean(rand_delta_p_means)),
        "top_minus_random_delta_retr": float(np.mean(delta_retr_top) - np.mean(rand_delta_r_means)),
        "bootstrap_mean_delta_p": mean_dp,
        "bootstrap_delta_p_ci_lo": dp_lo,
        "bootstrap_delta_p_ci_hi": dp_hi,
        "bootstrap_mean_delta_retr": mean_dr,
        "bootstrap_delta_retr_ci_lo": dr_lo,
        "bootstrap_delta_retr_ci_hi": dr_hi,
    }


all_tuning_rows = []
for term, ctx in primary_contexts.items():
    alpha_base = float(ctx["alphas"][1]) if len(ctx["alphas"]) > 1 else 0.2
    for k in SEARCH_GRID["k_latents"]:
        for m in SEARCH_GRID["alpha_mult"]:
            for d in SEARCH_GRID["direction"]:
                for efk in SEARCH_GRID["edit_first_k"]:
                    row = evaluate_setting(
                        ctx,
                        k_latents=k,
                        alpha=alpha_base * m,
                        direction=d,
                        edit_first_k=efk,
                        random_trials=PRIMARY_CONFIG["random_trials"],
                        top_n=PRIMARY_CONFIG["top_n_retrieval"],
                    )
                    all_tuning_rows.append(row)

term_tuning_df = pd.DataFrame(all_tuning_rows)
term_tuning_df.to_csv("primary_terms_tuning_grid.csv", index=False)
print("Saved primary_terms_tuning_grid.csv")
display(term_tuning_df.head())

# choose best setting by combined effect (probe-direction + retrieval-direction, top-random advantage)
term_tuning_df["selection_score"] = (
    term_tuning_df["top_minus_random_delta_p"] * 0.7
    + term_tuning_df["top_minus_random_delta_retr"] * 0.3
)

best_settings = term_tuning_df.sort_values("selection_score", ascending=False).groupby("term", as_index=False).first()
best_settings.to_csv("primary_terms_best_settings.csv", index=False)
print("Saved primary_terms_best_settings.csv")
display(best_settings[[
    "term", "k_latents", "alpha", "direction", "edit_first_k",
    "top_minus_random_delta_p", "top_minus_random_delta_retr",
    "bootstrap_delta_p_ci_lo", "bootstrap_delta_p_ci_hi",
    "bootstrap_delta_retr_ci_lo", "bootstrap_delta_retr_ci_hi",
    "selection_score",
]])

# shuffled-label sanity checks (probe-level)
shuffle_rows = []
for term, ctx in primary_contexts.items():
    labels_df = pd.read_csv(PRIMARY_LABEL_PATHS[term])
    data = manifest[["acc", "split"]].merge(labels_df[["acc", "protein_label"]], on="acc", how="inner")
    data = data.merge(embed_df_global[["acc", "embedding"]], on="acc", how="inner")
    data = data[data["protein_label"].isin([0, 1])].copy().reset_index(drop=True)

    X_all = np.stack(data["embedding"].to_list(), axis=0)
    y = data["protein_label"].astype(int).to_numpy()
    tr = data["split"].values == "train"
    va = data["split"].values == "val"
    te = data["split"].values == "test"

    Xtr, Xva, Xte = X_all[tr], X_all[va], X_all[te]
    ytr, yva, yte = y[tr], y[va], y[te]

    rng = np.random.default_rng(SEED)
    aucs = []
    for _ in range(30):
        ytr_shuf = ytr.copy()
        rng.shuffle(ytr_shuf)
        clf_shuf = LogisticRegression(max_iter=3000, class_weight="balanced")
        clf_shuf.fit(Xtr, ytr_shuf)
        p_te = clf_shuf.predict_proba(Xte)[:, 1]
        aucs.append(float(roc_auc_score(yte, p_te)) if len(np.unique(yte)) > 1 else np.nan)

    shuffle_rows.append({
        "term": term,
        "shuffle_auc_test_mean": float(np.nanmean(aucs)),
        "shuffle_auc_test_std": float(np.nanstd(aucs)),
        "n_trials": 30,
    })

shuffle_df = pd.DataFrame(shuffle_rows)
shuffle_df.to_csv("primary_terms_shuffled_label_sanity.csv", index=False)
print("Saved primary_terms_shuffled_label_sanity.csv")
display(shuffle_df)

In [ ]:
# Step 12D: final primary-claim report CSV (probe + tuned settings + controls + CIs)

probe_files = ["probe_metrics_disorder.csv", "probe_metrics_disorder_to_order.csv"]
probe_rows = []
for p in probe_files:
    if os.path.exists(p):
        probe_rows.append(pd.read_csv(p))
probe_all = pd.concat(probe_rows, ignore_index=True) if len(probe_rows) else pd.DataFrame()

if len(probe_all) == 0:
    raise ValueError("Missing probe metrics for primary terms. Run probe cells first.")

probe_pivot = probe_all.pivot(index="term", columns="split", values=["auc", "f1", "acc"])
probe_pivot.columns = [f"probe_{m}_{s}" for (m, s) in probe_pivot.columns]
probe_pivot = probe_pivot.reset_index()

best_settings = pd.read_csv("primary_terms_best_settings.csv") if os.path.exists("primary_terms_best_settings.csv") else pd.DataFrame()
shuffle_df = pd.read_csv("primary_terms_shuffled_label_sanity.csv") if os.path.exists("primary_terms_shuffled_label_sanity.csv") else pd.DataFrame()

report = probe_pivot.copy()
if len(best_settings):
    keep_cols = [
        "term", "k_latents", "alpha", "direction", "edit_first_k",
        "mean_delta_p_top", "mean_delta_retr_top",
        "mean_delta_p_random", "mean_delta_retr_random",
        "mean_delta_p_signflip", "mean_delta_retr_signflip",
        "top_minus_random_delta_p", "top_minus_random_delta_retr",
        "bootstrap_delta_p_ci_lo", "bootstrap_delta_p_ci_hi",
        "bootstrap_delta_retr_ci_lo", "bootstrap_delta_retr_ci_hi",
        "selection_score",
    ]
    keep_cols = [c for c in keep_cols if c in best_settings.columns]
    report = report.merge(best_settings[keep_cols], on="term", how="left")

if len(shuffle_df):
    report = report.merge(shuffle_df, on="term", how="left")

# lightweight quality flags
report["flag_probe_good"] = report.get("probe_auc_test", np.nan) >= 0.65
report["flag_edit_beats_random"] = report.get("top_minus_random_delta_p", np.nan) > 0
report["flag_retrieval_positive"] = report.get("mean_delta_retr_top", np.nan) > 0

report = report.sort_values("term").reset_index(drop=True)
report.to_csv("primary_claim_report.csv", index=False)
print("Saved primary_claim_report.csv")
display(report)

# compact table for quick readout
compact_cols = [
    "term", "probe_auc_test", "probe_f1_test",
    "k_latents", "alpha", "direction", "edit_first_k",
    "top_minus_random_delta_p", "top_minus_random_delta_retr",
    "shuffle_auc_test_mean",
    "flag_probe_good", "flag_edit_beats_random", "flag_retrieval_positive",
]
compact_cols = [c for c in compact_cols if c in report.columns]
compact = report[compact_cols].copy()
compact.to_csv("primary_claim_report_compact.csv", index=False)
print("Saved primary_claim_report_compact.csv")
display(compact)

In [ ]:
# Step 12C-fast: lighter tuning/controls version (use this if original 12C is too slow)

import time
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity

FAST_NEAR_N = 60
FAST_RANDOM_TRIALS = 3
FAST_BOOTSTRAP_N = 250
FAST_SHUFFLE_TRIALS = 10
FAST_GRID = {
    "k_latents": [1, 5],
    "alpha_mult": [1.0, 2.0],
    "direction": [-1, 1],
    "edit_first_k": [30],
}

print("Running 12C-fast with settings:")
print({
    "near_n": FAST_NEAR_N,
    "random_trials": FAST_RANDOM_TRIALS,
    "bootstrap_n": FAST_BOOTSTRAP_N,
    "shuffle_trials": FAST_SHUFFLE_TRIALS,
    "grid": FAST_GRID,
})


def evaluate_setting_fast(ctx, k_latents, alpha, direction, edit_first_k, random_trials=3, top_n=10, near_n=60):
    clf_term = ctx["clf"]
    test_df = ctx["test_near"].head(min(near_n, len(ctx["test_near"]))).copy()
    bank_emb = ctx["bank_emb"]
    bank_y = ctx["bank_y"]
    lat_idx = ctx["ranked_latents"][:int(k_latents)]
    alpha_eff = float(alpha) * int(direction)

    def _prob(x):
        return float(clf_term.predict_proba(x.reshape(1, -1))[0, 1])

    def _nn(query):
        sims = cosine_similarity(query.reshape(1, -1), bank_emb)[0]
        idx = np.argsort(-sims)[:top_n]
        return idx

    delta_p_top, delta_retr_top = [], []
    seqs = test_df["sequence"].tolist()

    for seq in seqs:
        seq = str(seq)[:SEQ_TRUNC]
        x0 = seq_to_orig_emb_term(seq)
        x1 = seq_to_edit_emb_term(seq, lat_idx, alpha_eff, edit_first_k)
        p0, p1 = _prob(x0), _prob(x1)
        i0, i1 = _nn(x0), _nn(x1)
        delta_p_top.append(p1 - p0)
        delta_retr_top.append(float(bank_y[i1].mean() - bank_y[i0].mean()))

    # random control
    rng = np.random.default_rng(SEED)
    d_lat = int(sae.enc.out_features)
    rand_dp, rand_dr = [], []
    for _ in range(random_trials):
        r_idx = rng.choice(d_lat, size=int(k_latents), replace=False)
        dp_tmp, dr_tmp = [], []
        for seq in seqs:
            seq = str(seq)[:SEQ_TRUNC]
            x0 = seq_to_orig_emb_term(seq)
            x1 = seq_to_edit_emb_term(seq, r_idx, alpha_eff, edit_first_k)
            p0, p1 = _prob(x0), _prob(x1)
            i0, i1 = _nn(x0), _nn(x1)
            dp_tmp.append(p1 - p0)
            dr_tmp.append(float(bank_y[i1].mean() - bank_y[i0].mean()))
        rand_dp.append(float(np.mean(dp_tmp)))
        rand_dr.append(float(np.mean(dr_tmp)))

    # sign-flip control
    sf_dp, sf_dr = [], []
    for seq in seqs:
        seq = str(seq)[:SEQ_TRUNC]
        x0 = seq_to_orig_emb_term(seq)
        x1 = seq_to_edit_emb_term(seq, lat_idx, -alpha_eff, edit_first_k)
        p0, p1 = _prob(x0), _prob(x1)
        i0, i1 = _nn(x0), _nn(x1)
        sf_dp.append(p1 - p0)
        sf_dr.append(float(bank_y[i1].mean() - bank_y[i0].mean()))

    mean_dp, dp_lo, dp_hi = bootstrap_ci(delta_p_top, n_boot=FAST_BOOTSTRAP_N, seed=SEED)
    mean_dr, dr_lo, dr_hi = bootstrap_ci(delta_retr_top, n_boot=FAST_BOOTSTRAP_N, seed=SEED)

    return {
        "term": ctx["term"],
        "k_latents": int(k_latents),
        "alpha": float(alpha),
        "direction": int(direction),
        "edit_first_k": int(edit_first_k),
        "n_eval": int(len(seqs)),
        "mean_delta_p_top": float(np.mean(delta_p_top)),
        "mean_delta_retr_top": float(np.mean(delta_retr_top)),
        "mean_delta_p_random": float(np.mean(rand_dp)),
        "mean_delta_retr_random": float(np.mean(rand_dr)),
        "mean_delta_p_signflip": float(np.mean(sf_dp)),
        "mean_delta_retr_signflip": float(np.mean(sf_dr)),
        "top_minus_random_delta_p": float(np.mean(delta_p_top) - np.mean(rand_dp)),
        "top_minus_random_delta_retr": float(np.mean(delta_retr_top) - np.mean(rand_dr)),
        "bootstrap_mean_delta_p": mean_dp,
        "bootstrap_delta_p_ci_lo": dp_lo,
        "bootstrap_delta_p_ci_hi": dp_hi,
        "bootstrap_mean_delta_retr": mean_dr,
        "bootstrap_delta_retr_ci_lo": dr_lo,
        "bootstrap_delta_retr_ci_hi": dr_hi,
    }

start = time.time()
rows = []

for term, ctx in primary_contexts.items():
    alpha_base = float(ctx["alphas"][1]) if len(ctx["alphas"]) > 1 else 0.2
    jobs = (
        len(FAST_GRID["k_latents"]) * len(FAST_GRID["alpha_mult"]) *
        len(FAST_GRID["direction"]) * len(FAST_GRID["edit_first_k"])
    )
    done = 0
    print(f"\n[{term}] running {jobs} jobs...")

    for k in FAST_GRID["k_latents"]:
        for m in FAST_GRID["alpha_mult"]:
            for d in FAST_GRID["direction"]:
                for efk in FAST_GRID["edit_first_k"]:
                    row = evaluate_setting_fast(
                        ctx,
                        k_latents=k,
                        alpha=alpha_base * m,
                        direction=d,
                        edit_first_k=efk,
                        random_trials=FAST_RANDOM_TRIALS,
                        top_n=PRIMARY_CONFIG["top_n_retrieval"],
                        near_n=FAST_NEAR_N,
                    )
                    rows.append(row)
                    done += 1
                    if done % 2 == 0:
                        print(f"[{term}] progress: {done}/{jobs}")

term_tuning_df = pd.DataFrame(rows)
term_tuning_df.to_csv("primary_terms_tuning_grid.csv", index=False)
print("Saved primary_terms_tuning_grid.csv")

term_tuning_df["selection_score"] = (
    term_tuning_df["top_minus_random_delta_p"] * 0.7
    + term_tuning_df["top_minus_random_delta_retr"] * 0.3
)
best_settings = term_tuning_df.sort_values("selection_score", ascending=False).groupby("term", as_index=False).first()
best_settings.to_csv("primary_terms_best_settings.csv", index=False)
print("Saved primary_terms_best_settings.csv")
display(best_settings[[
    "term", "k_latents", "alpha", "direction", "edit_first_k",
    "top_minus_random_delta_p", "top_minus_random_delta_retr",
    "bootstrap_delta_p_ci_lo", "bootstrap_delta_p_ci_hi",
    "bootstrap_delta_retr_ci_lo", "bootstrap_delta_retr_ci_hi",
    "selection_score",
]])

# shuffled-label sanity (lighter)
shuffle_rows = []
for term in PRIMARY_TERMS:
    labels_df = pd.read_csv(PRIMARY_LABEL_PATHS[term])
    data = manifest[["acc", "split"]].merge(labels_df[["acc", "protein_label"]], on="acc", how="inner")
    data = data.merge(embed_df_global[["acc", "embedding"]], on="acc", how="inner")
    data = data[data["protein_label"].isin([0, 1])].copy().reset_index(drop=True)

    X_all = np.stack(data["embedding"].to_list(), axis=0)
    y = data["protein_label"].astype(int).to_numpy()
    tr = data["split"].values == "train"
    te = data["split"].values == "test"

    Xtr, Xte = X_all[tr], X_all[te]
    ytr, yte = y[tr], y[te]

    rng = np.random.default_rng(SEED)
    aucs = []
    for _ in range(FAST_SHUFFLE_TRIALS):
        ytr_s = ytr.copy()
        rng.shuffle(ytr_s)
        clf_s = LogisticRegression(max_iter=3000, class_weight="balanced")
        clf_s.fit(Xtr, ytr_s)
        p_te = clf_s.predict_proba(Xte)[:, 1]
        aucs.append(float(roc_auc_score(yte, p_te)) if len(np.unique(yte)) > 1 else np.nan)

    shuffle_rows.append({
        "term": term,
        "shuffle_auc_test_mean": float(np.nanmean(aucs)),
        "shuffle_auc_test_std": float(np.nanstd(aucs)),
        "n_trials": FAST_SHUFFLE_TRIALS,
    })

shuffle_df = pd.DataFrame(shuffle_rows)
shuffle_df.to_csv("primary_terms_shuffled_label_sanity.csv", index=False)
print("Saved primary_terms_shuffled_label_sanity.csv")
display(shuffle_df)

print(f"12C-fast elapsed: {(time.time() - start)/60:.1f} min")

In [ ]:
# Step 13A: retrieval-improvement setup (primary terms only, quantile near-boundary, fixed seeds)

import os
import math
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

if "primary_contexts" not in globals() or len(primary_contexts) == 0:
    raise ValueError("primary_contexts is missing. Run Step 12B first.")

RETRIEVAL_IMPROVEMENT_CONFIG = {
    "seed": int(SEED),
    "terms": ["disorder", "disorder_to_order"],
    "k_latents": [1, 5, 10, 20],
    "alpha_mult": [0.5, 1.0, 2.0, 4.0, 8.0],
    "direction": [-1, 1],
    "edit_first_k": [10, 30, 60, 120],
    "top_n_retrieval": int(PRIMARY_CONFIG.get("top_n_retrieval", 10)),
    "random_trials": 8,
    "shuffled_label_trials": 60,
    "bootstrap_n": 1200,
    "near_boundary_quantile": 0.20,
    "near_boundary_min_n": 60,
}

RETRIEVAL_OUTPUT_DIR = "/Users/atifmomin/Downloads/intermediate data"
RETRIEVAL_REPORT_PATH = os.path.join(RETRIEVAL_OUTPUT_DIR, "retrieval_improvement_report.csv")
os.makedirs(RETRIEVAL_OUTPUT_DIR, exist_ok=True)


def _safe_nanmean(values):
    arr = np.asarray(values, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    return float(np.mean(arr))


def _softmax(x):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        return x
    x = x - np.max(x)
    e = np.exp(x)
    s = np.sum(e)
    if s <= 0:
        return np.ones_like(x) / float(len(x))
    return e / s


def bootstrap_ci(values, n_boot=1000, seed=42, alpha=0.05):
    vals = np.asarray(values, dtype=np.float64)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(int(seed))
    means = np.empty(int(n_boot), dtype=np.float64)
    for i in range(int(n_boot)):
        sample = rng.choice(vals, size=vals.size, replace=True)
        means[i] = np.mean(sample)
    return float(np.mean(vals)), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))


def bootstrap_diff_ci(a, b, n_boot=1000, seed=42, alpha=0.05):
    aa = np.asarray(a, dtype=np.float64)
    bb = np.asarray(b, dtype=np.float64)
    mask = np.isfinite(aa) & np.isfinite(bb)
    aa, bb = aa[mask], bb[mask]
    if aa.size == 0:
        return np.nan, np.nan, np.nan
    diff = aa - bb
    return bootstrap_ci(diff, n_boot=n_boot, seed=seed, alpha=alpha)


def select_eval_subsets(ctx, q=0.20, min_n=60):
    test_df = ctx["test_set"].copy().reset_index(drop=True)
    if len(test_df) == 0:
        return {"global_test": test_df.copy(), "near_boundary": test_df.copy()}

    X = np.stack(test_df["embedding"].to_list(), axis=0)
    p = ctx["clf"].predict_proba(X)[:, 1]
    proximity = np.abs(p - 0.5)
    n_q = int(math.ceil(float(q) * len(test_df)))
    n_keep = int(max(int(min_n), n_q))
    n_keep = int(min(n_keep, len(test_df)))
    idx = np.argsort(proximity)[:n_keep]
    near_df = test_df.iloc[idx].copy().reset_index(drop=True)

    return {
        "global_test": test_df,
        "near_boundary": near_df,
    }


print("Retrieval improvement setup ready:")
print({
    "terms": RETRIEVAL_IMPROVEMENT_CONFIG["terms"],
    "k_latents": RETRIEVAL_IMPROVEMENT_CONFIG["k_latents"],
    "alpha_mult": RETRIEVAL_IMPROVEMENT_CONFIG["alpha_mult"],
    "edit_first_k": RETRIEVAL_IMPROVEMENT_CONFIG["edit_first_k"],
    "near_boundary_quantile": RETRIEVAL_IMPROVEMENT_CONFIG["near_boundary_quantile"],
    "output": RETRIEVAL_REPORT_PATH,
})

In [ ]:
# Step 13B: expanded retrieval metrics + controls per setting


def _retrieval_metrics_from_neighbors(i0, s0, i1, s1, bank_y):
    y0 = bank_y[i0]
    y1 = bank_y[i1]

    delta_retr = float(np.mean(y1) - np.mean(y0))

    set0 = set(map(int, np.asarray(i0).tolist()))
    set1 = set(map(int, np.asarray(i1).tolist()))
    union = set0 | set1
    inter = set0 & set1
    jacc = float(len(inter) / len(union)) if len(union) else np.nan

    entered = list(set1 - set0)
    if len(entered):
        y_new = bank_y[np.asarray(entered, dtype=np.int64)]
        new_pos_count = float(np.sum(y_new == 1))
        new_label_mean = float(np.mean(y_new))
    else:
        new_pos_count = 0.0
        new_label_mean = np.nan

    w0 = _softmax(np.asarray(s0, dtype=np.float64))
    w1 = _softmax(np.asarray(s1, dtype=np.float64))
    score0 = float(np.sum(w0 * y0)) if len(w0) else np.nan
    score1 = float(np.sum(w1 * y1)) if len(w1) else np.nan
    weighted_shift = float(score1 - score0) if np.isfinite(score0) and np.isfinite(score1) else np.nan

    return {
        "delta_retr": delta_retr,
        "jaccard_neighbor_overlap": jacc,
        "jaccard_change": float(1.0 - jacc) if np.isfinite(jacc) else np.nan,
        "new_positive_count": new_pos_count,
        "new_neighbor_label": new_label_mean,
        "weighted_knn_label_shift": weighted_shift,
    }


def _mean_of_metrics_dict(metrics_dict):
    return {f"mean_{k}": _safe_nanmean(v) for k, v in metrics_dict.items()}


def _setting_seed(base_seed, term, subset_name, k_latents, alpha, direction, edit_first_k):
    term_key = sum(ord(c) for c in str(term))
    subset_key = sum(ord(c) for c in str(subset_name))
    alpha_key = int(round(float(alpha) * 10000.0))
    value = (
        int(base_seed) * 31
        + int(term_key) * 17
        + int(subset_key) * 13
        + int(k_latents) * 11
        + int(direction) * 7
        + int(edit_first_k) * 5
        + int(alpha_key)
    )
    return int(value % (2**32 - 1))


def evaluate_retrieval_setting(
    ctx,
    subset_name,
    subset_df,
    k_latents,
    alpha,
    direction,
    edit_first_k,
    top_n,
    random_trials,
    shuffled_trials,
    bootstrap_n,
    seed,
):
    term = ctx["term"]
    clf_term = ctx["clf"]
    bank_emb = ctx["bank_emb"]
    bank_y = np.asarray(ctx["bank_y"], dtype=np.int64)
    lat_idx = np.asarray(ctx["ranked_latents"][:int(k_latents)], dtype=np.int64)
    alpha_eff = float(alpha) * int(direction)

    seqs = [str(s)[:SEQ_TRUNC] for s in subset_df["sequence"].tolist()]
    n_eval = len(seqs)

    if n_eval == 0:
        return {
            "term": term,
            "subset": subset_name,
            "k_latents": int(k_latents),
            "alpha": float(alpha),
            "direction": int(direction),
            "edit_first_k": int(edit_first_k),
            "n_eval": 0,
        }

    def _prob(x):
        return float(clf_term.predict_proba(x.reshape(1, -1))[0, 1])

    def _neighbors(query):
        sims = cosine_similarity(query.reshape(1, -1), bank_emb)[0]
        idx = np.argsort(-sims)[:int(top_n)]
        return idx, sims[idx]

    metric_keys = [
        "delta_retr",
        "jaccard_neighbor_overlap",
        "jaccard_change",
        "new_positive_count",
        "new_neighbor_label",
        "weighted_knn_label_shift",
    ]

    top_probe_delta = []
    top_metrics = {k: [] for k in metric_keys}
    top_i0, top_i1 = [], []

    for seq in seqs:
        x0 = seq_to_orig_emb_term(seq)
        x1 = seq_to_edit_emb_term(seq, lat_idx, alpha_eff, edit_first_k)
        p0, p1 = _prob(x0), _prob(x1)
        i0, s0 = _neighbors(x0)
        i1, s1 = _neighbors(x1)

        top_probe_delta.append(float(p1 - p0))
        md = _retrieval_metrics_from_neighbors(i0, s0, i1, s1, bank_y)
        for k in metric_keys:
            top_metrics[k].append(md[k])
        top_i0.append(i0)
        top_i1.append(i1)

    setting_seed = _setting_seed(seed, term, subset_name, k_latents, alpha, direction, edit_first_k)
    rng = np.random.default_rng(setting_seed)
    d_lat = int(sae.enc.out_features)

    rand_probe_trials = []
    rand_metric_trials = []
    for _ in range(int(random_trials)):
        r_idx = rng.choice(d_lat, size=int(k_latents), replace=False)
        rp = []
        rm = {k: [] for k in metric_keys}
        for seq in seqs:
            x0 = seq_to_orig_emb_term(seq)
            xr = seq_to_edit_emb_term(seq, r_idx, alpha_eff, edit_first_k)
            p0, pr = _prob(x0), _prob(xr)
            i0, s0 = _neighbors(x0)
            ir, sr = _neighbors(xr)
            rp.append(float(pr - p0))
            md = _retrieval_metrics_from_neighbors(i0, s0, ir, sr, bank_y)
            for k in metric_keys:
                rm[k].append(md[k])
        rand_probe_trials.append(np.asarray(rp, dtype=np.float64))
        rand_metric_trials.append({k: np.asarray(v, dtype=np.float64) for k, v in rm.items()})

    if len(rand_probe_trials):
        rand_probe_mean_per_sample = np.nanmean(np.stack(rand_probe_trials, axis=0), axis=0)
        rand_metrics_mean_per_sample = {
            k: np.nanmean(np.stack([t[k] for t in rand_metric_trials], axis=0), axis=0)
            for k in metric_keys
        }
    else:
        rand_probe_mean_per_sample = np.full(n_eval, np.nan, dtype=np.float64)
        rand_metrics_mean_per_sample = {k: np.full(n_eval, np.nan, dtype=np.float64) for k in metric_keys}

    sf_probe_delta = []
    sf_metrics = {k: [] for k in metric_keys}
    for seq in seqs:
        x0 = seq_to_orig_emb_term(seq)
        xs = seq_to_edit_emb_term(seq, lat_idx, -alpha_eff, edit_first_k)
        p0, ps = _prob(x0), _prob(xs)
        i0, s0 = _neighbors(x0)
        isf, ssf = _neighbors(xs)
        sf_probe_delta.append(float(ps - p0))
        md = _retrieval_metrics_from_neighbors(i0, s0, isf, ssf, bank_y)
        for k in metric_keys:
            sf_metrics[k].append(md[k])

    shuffled_trial_means = []
    for _ in range(int(shuffled_trials)):
        y_shuf = bank_y.copy()
        rng.shuffle(y_shuf)
        vals = []
        for i0, i1 in zip(top_i0, top_i1):
            vals.append(float(np.mean(y_shuf[i1]) - np.mean(y_shuf[i0])))
        shuffled_trial_means.append(_safe_nanmean(vals))

    row = {
        "term": term,
        "subset": subset_name,
        "k_latents": int(k_latents),
        "alpha": float(alpha),
        "direction": int(direction),
        "edit_first_k": int(edit_first_k),
        "n_eval": int(n_eval),
    }

    top_probe_arr = np.asarray(top_probe_delta, dtype=np.float64)
    rand_probe_arr = np.asarray(rand_probe_mean_per_sample, dtype=np.float64)
    sf_probe_arr = np.asarray(sf_probe_delta, dtype=np.float64)

    row["mean_delta_p_top"] = _safe_nanmean(top_probe_arr)
    row["mean_delta_p_random"] = _safe_nanmean(rand_probe_arr)
    row["mean_delta_p_signflip"] = _safe_nanmean(sf_probe_arr)
    row["top_minus_random_delta_p"] = row["mean_delta_p_top"] - row["mean_delta_p_random"] if np.isfinite(row["mean_delta_p_top"]) and np.isfinite(row["mean_delta_p_random"]) else np.nan
    row["top_minus_signflip_delta_p"] = row["mean_delta_p_top"] - row["mean_delta_p_signflip"] if np.isfinite(row["mean_delta_p_top"]) and np.isfinite(row["mean_delta_p_signflip"]) else np.nan

    mean_top_metrics = _mean_of_metrics_dict(top_metrics)
    mean_sf_metrics = _mean_of_metrics_dict(sf_metrics)
    mean_rand_metrics = {f"mean_{k}": _safe_nanmean(v) for k, v in rand_metrics_mean_per_sample.items()}

    for key, val in mean_top_metrics.items():
        row[f"{key}_top"] = val
    for key, val in mean_rand_metrics.items():
        row[f"{key}_random"] = val
    for key, val in mean_sf_metrics.items():
        row[f"{key}_signflip"] = val

    for metric in metric_keys:
        top_name = f"mean_{metric}_top"
        rand_name = f"mean_{metric}_random"
        sf_name = f"mean_{metric}_signflip"
        row[f"top_minus_random_{top_name}"] = row[top_name] - row[rand_name] if np.isfinite(row[top_name]) and np.isfinite(row[rand_name]) else np.nan
        row[f"top_minus_signflip_{top_name}"] = row[top_name] - row[sf_name] if np.isfinite(row[top_name]) and np.isfinite(row[sf_name]) else np.nan

    mean_dp, dp_lo, dp_hi = bootstrap_ci(top_probe_arr, n_boot=bootstrap_n, seed=setting_seed + 1)
    row["bootstrap_mean_delta_p"] = mean_dp
    row["bootstrap_delta_p_ci_lo"] = dp_lo
    row["bootstrap_delta_p_ci_hi"] = dp_hi

    dr_top = np.asarray(top_metrics["delta_retr"], dtype=np.float64)
    mean_dr, dr_lo, dr_hi = bootstrap_ci(dr_top, n_boot=bootstrap_n, seed=setting_seed + 2)
    row["bootstrap_mean_delta_retr"] = mean_dr
    row["bootstrap_delta_retr_ci_lo"] = dr_lo
    row["bootstrap_delta_retr_ci_hi"] = dr_hi

    dpr_mean, dpr_lo, dpr_hi = bootstrap_diff_ci(top_probe_arr, rand_probe_arr, n_boot=bootstrap_n, seed=setting_seed + 3)
    row["bootstrap_top_minus_random_delta_p"] = dpr_mean
    row["bootstrap_top_minus_random_delta_p_ci_lo"] = dpr_lo
    row["bootstrap_top_minus_random_delta_p_ci_hi"] = dpr_hi

    dr_rand = np.asarray(rand_metrics_mean_per_sample["delta_retr"], dtype=np.float64)
    drr_mean, drr_lo, drr_hi = bootstrap_diff_ci(dr_top, dr_rand, n_boot=bootstrap_n, seed=setting_seed + 4)
    row["bootstrap_top_minus_random_delta_retr"] = drr_mean
    row["bootstrap_top_minus_random_delta_retr_ci_lo"] = drr_lo
    row["bootstrap_top_minus_random_delta_retr_ci_hi"] = drr_hi

    dr_sf = np.asarray(sf_metrics["delta_retr"], dtype=np.float64)
    drs_mean, drs_lo, drs_hi = bootstrap_diff_ci(dr_top, dr_sf, n_boot=bootstrap_n, seed=setting_seed + 5)
    row["bootstrap_top_minus_signflip_delta_retr"] = drs_mean
    row["bootstrap_top_minus_signflip_delta_retr_ci_lo"] = drs_lo
    row["bootstrap_top_minus_signflip_delta_retr_ci_hi"] = drs_hi

    shuf = np.asarray(shuffled_trial_means, dtype=np.float64)
    row["shuffled_retr_delta_mean"] = _safe_nanmean(shuf)
    row["shuffled_retr_delta_std"] = float(np.nanstd(shuf)) if np.isfinite(_safe_nanmean(shuf)) else np.nan
    row["shuffled_retr_trials"] = int(len(shuf))

    obs = row["mean_delta_retr_top"]
    if np.isfinite(obs) and len(shuf):
        row["shuffled_retr_p_ge_obs"] = float((1.0 + np.sum(shuf >= obs)) / (len(shuf) + 1.0))
        denom = float(np.nanstd(shuf)) + 1e-8
        row["shuffled_retr_delta_z"] = float((obs - np.nanmean(shuf)) / denom)
    else:
        row["shuffled_retr_p_ge_obs"] = np.nan
        row["shuffled_retr_delta_z"] = np.nan

    return row

In [ ]:
# Step 13C: run retrieval-improvement sweep and write retrieval_improvement_report.csv

import time

# Set RUN_MODE to "quick" for iteration (target: ~30-90 min), "full" for final pass.
RUN_MODE = "quick"  # change to "full" when ready

if RUN_MODE not in {"quick", "full"}:
    raise ValueError("RUN_MODE must be 'quick' or 'full'")

# Quick-mode presets keep methodology identical, just fewer settings/trials.
if RUN_MODE == "quick":
    active_grid = {
        "k_latents": [1, 5, 10],
        "alpha_mult": [1.0, 2.0],
        "direction": [-1, 1],
        "edit_first_k": [30, 60],
    }
    active_random_trials = 3
    active_shuffled_trials = 15
    active_bootstrap_n = 250
    active_subsets = ["near_boundary", "global_test"]
else:
    active_grid = {
        "k_latents": RETRIEVAL_IMPROVEMENT_CONFIG["k_latents"],
        "alpha_mult": RETRIEVAL_IMPROVEMENT_CONFIG["alpha_mult"],
        "direction": RETRIEVAL_IMPROVEMENT_CONFIG["direction"],
        "edit_first_k": RETRIEVAL_IMPROVEMENT_CONFIG["edit_first_k"],
    }
    active_random_trials = RETRIEVAL_IMPROVEMENT_CONFIG["random_trials"]
    active_shuffled_trials = RETRIEVAL_IMPROVEMENT_CONFIG["shuffled_label_trials"]
    active_bootstrap_n = RETRIEVAL_IMPROVEMENT_CONFIG["bootstrap_n"]
    active_subsets = ["global_test", "near_boundary"]

rows = []
terms_to_run = [t for t in RETRIEVAL_IMPROVEMENT_CONFIG["terms"] if t in primary_contexts]

jobs_per_term = (
    len(active_subsets)
    * len(active_grid["k_latents"])
    * len(active_grid["alpha_mult"])
    * len(active_grid["direction"])
    * len(active_grid["edit_first_k"])
)
total_jobs_global = jobs_per_term * len(terms_to_run)

print("Step 13C run config:")
print({
    "run_mode": RUN_MODE,
    "terms": terms_to_run,
    "jobs_per_term": jobs_per_term,
    "jobs_total": total_jobs_global,
    "random_trials": active_random_trials,
    "shuffled_trials": active_shuffled_trials,
    "bootstrap_n": active_bootstrap_n,
    "grid": active_grid,
})

start_global = time.time()
done_global = 0

for term in terms_to_run:
    ctx = primary_contexts[term]
    subsets_all = select_eval_subsets(
        ctx,
        q=RETRIEVAL_IMPROVEMENT_CONFIG["near_boundary_quantile"],
        min_n=RETRIEVAL_IMPROVEMENT_CONFIG["near_boundary_min_n"],
    )
    alpha_base = float(ctx["alphas"][1]) if len(ctx["alphas"]) > 1 else 0.2

    done_term = 0
    start_term = time.time()
    print(f"[{term}] running {jobs_per_term} settings...")

    for subset_name in active_subsets:
        subset_df = subsets_all[subset_name]
        for k in active_grid["k_latents"]:
            for mult in active_grid["alpha_mult"]:
                for direction in active_grid["direction"]:
                    for efk in active_grid["edit_first_k"]:
                        row = evaluate_retrieval_setting(
                            ctx=ctx,
                            subset_name=subset_name,
                            subset_df=subset_df,
                            k_latents=k,
                            alpha=alpha_base * mult,
                            direction=direction,
                            edit_first_k=efk,
                            top_n=RETRIEVAL_IMPROVEMENT_CONFIG["top_n_retrieval"],
                            random_trials=active_random_trials,
                            shuffled_trials=active_shuffled_trials,
                            bootstrap_n=active_bootstrap_n,
                            seed=RETRIEVAL_IMPROVEMENT_CONFIG["seed"],
                        )
                        rows.append(row)
                        done_term += 1
                        done_global += 1

                        if done_term % 8 == 0 or done_term == jobs_per_term:
                            elapsed_term = max(1e-6, time.time() - start_term)
                            rate_term = done_term / elapsed_term
                            rem_term = jobs_per_term - done_term
                            eta_term_min = rem_term / max(rate_term, 1e-6) / 60.0

                            elapsed_global = max(1e-6, time.time() - start_global)
                            rate_global = done_global / elapsed_global
                            rem_global = total_jobs_global - done_global
                            eta_global_min = rem_global / max(rate_global, 1e-6) / 60.0

                            print(
                                f"[{term}] {done_term}/{jobs_per_term} | "
                                f"ETA_term={eta_term_min:.1f}m | ETA_total={eta_global_min:.1f}m"
                            )

retrieval_improvement_df = pd.DataFrame(rows)
if len(retrieval_improvement_df) == 0:
    raise ValueError("No retrieval-improvement rows were generated.")

# retrieval-first ranking; probe shift is a light tie-breaker
retrieval_improvement_df["retrieval_score"] = (
    2.5 * retrieval_improvement_df["top_minus_random_mean_delta_retr_top"].fillna(0.0)
    + 1.5 * retrieval_improvement_df["top_minus_signflip_mean_delta_retr_top"].fillna(0.0)
    + 0.8 * retrieval_improvement_df["top_minus_random_mean_new_positive_count_top"].fillna(0.0)
    + 0.6 * retrieval_improvement_df["top_minus_random_mean_new_neighbor_label_top"].fillna(0.0)
    + 0.8 * retrieval_improvement_df["top_minus_random_mean_weighted_knn_label_shift_top"].fillna(0.0)
    + 0.9 * retrieval_improvement_df["bootstrap_top_minus_random_delta_retr_ci_lo"].fillna(0.0)
    + 0.5 * retrieval_improvement_df["bootstrap_top_minus_signflip_delta_retr_ci_lo"].fillna(0.0)
    + 0.3 * retrieval_improvement_df["shuffled_retr_delta_z"].fillna(0.0)
    + 0.05 * retrieval_improvement_df["top_minus_random_delta_p"].fillna(0.0)
)

retrieval_improvement_df = retrieval_improvement_df.sort_values(
    ["term", "retrieval_score", "top_minus_random_delta_p"],
    ascending=[True, False, False],
).reset_index(drop=True)

retrieval_improvement_df["run_mode"] = RUN_MODE
retrieval_improvement_df["rank_within_term"] = retrieval_improvement_df.groupby("term").cumcount() + 1
retrieval_improvement_df["rank_within_term_subset"] = retrieval_improvement_df.groupby(["term", "subset"]).cumcount() + 1
retrieval_improvement_df["is_term_best"] = retrieval_improvement_df["rank_within_term"] == 1
retrieval_improvement_df["is_term_subset_best"] = retrieval_improvement_df["rank_within_term_subset"] == 1

retrieval_improvement_df.to_csv(RETRIEVAL_REPORT_PATH, index=False)
print(f"Saved {RETRIEVAL_REPORT_PATH}")
print(f"Total elapsed: {(time.time() - start_global)/60.0:.1f} min")

preview_cols = [
    "term", "subset", "k_latents", "alpha", "direction", "edit_first_k",
    "n_eval", "mean_delta_p_top", "mean_delta_retr_top",
    "top_minus_random_delta_p", "top_minus_random_mean_delta_retr_top",
    "mean_jaccard_neighbor_overlap_top", "mean_new_positive_count_top",
    "mean_new_neighbor_label_top", "mean_weighted_knn_label_shift_top",
    "bootstrap_top_minus_random_delta_retr_ci_lo", "bootstrap_top_minus_random_delta_retr_ci_hi",
    "shuffled_retr_p_ge_obs", "retrieval_score", "rank_within_term", "is_term_best",
]
preview_cols = [c for c in preview_cols if c in retrieval_improvement_df.columns]
display(retrieval_improvement_df[preview_cols].head(20))

In [ ]:
# Step 13D: quick validation checks for coverage/determinism/signs

if not os.path.exists(RETRIEVAL_REPORT_PATH):
    raise FileNotFoundError(f"Missing {RETRIEVAL_REPORT_PATH}")

ri = pd.read_csv(RETRIEVAL_REPORT_PATH)

required_terms = set(RETRIEVAL_IMPROVEMENT_CONFIG["terms"])
found_terms = set(ri["term"].dropna().unique().tolist())
missing_terms = required_terms - found_terms

required_subsets = {"global_test", "near_boundary"}
subset_check = ri.groupby("term")["subset"].apply(lambda s: set(s.dropna().unique().tolist())).to_dict()

print("Rows:", len(ri))
print("Terms found:", sorted(found_terms))
print("Missing terms:", sorted(missing_terms))
print("Subset coverage by term:", subset_check)

if len(missing_terms) > 0:
    raise AssertionError(f"Missing term rows in retrieval report: {missing_terms}")

for t in required_terms:
    missing_subsets = required_subsets - subset_check.get(t, set())
    if missing_subsets:
        raise AssertionError(f"[{t}] missing subsets: {missing_subsets}")

# deterministic readback check
ri2 = pd.read_csv(RETRIEVAL_REPORT_PATH)
if len(ri) != len(ri2):
    raise AssertionError("Readback row-count mismatch")

# sign-oriented snapshot for practical decision-making
snapshot_cols = [
    "term", "subset", "rank_within_term", "k_latents", "alpha", "direction", "edit_first_k",
    "top_minus_random_mean_delta_retr_top", "bootstrap_top_minus_random_delta_retr_ci_lo",
    "top_minus_signflip_mean_delta_retr_top", "mean_new_positive_count_top",
    "mean_weighted_knn_label_shift_top", "top_minus_random_delta_p", "retrieval_score",
]
snapshot_cols = [c for c in snapshot_cols if c in ri.columns]
best_snapshot = ri.sort_values(["term", "retrieval_score"], ascending=[True, False]).groupby("term", as_index=False).first()
print("Best rows per term:")
display(best_snapshot[snapshot_cols])

print("Validation checks passed.")

In [ ]:
# Step 13E: refine pass on top quick finalists with stronger controls/CIs

import time

REFINE_TOP_K_PER_TERM = 8  # choose 5-10 as needed
REFINE_CONFIG = {
    "random_trials": 8,
    "shuffled_trials": 40,
    "bootstrap_n": 1000,
    "seed": int(RETRIEVAL_IMPROVEMENT_CONFIG["seed"]),
}

REFINE_REPORT_PATH = os.path.join(RETRIEVAL_OUTPUT_DIR, "retrieval_improvement_report_refine.csv")

if not os.path.exists(RETRIEVAL_REPORT_PATH):
    raise FileNotFoundError(f"Quick report missing: {RETRIEVAL_REPORT_PATH}. Run Step 13C quick first.")

quick_df = pd.read_csv(RETRIEVAL_REPORT_PATH)
if len(quick_df) == 0:
    raise ValueError("Quick report is empty.")

if "run_mode" in quick_df.columns:
    quick_df = quick_df[quick_df["run_mode"].fillna("quick") == "quick"].copy()

if "retrieval_score" not in quick_df.columns:
    raise ValueError("Quick report missing retrieval_score column.")

quick_df = quick_df.sort_values(["term", "retrieval_score", "top_minus_random_delta_p"], ascending=[True, False, False])
finalists = quick_df.groupby("term", as_index=False).head(int(REFINE_TOP_K_PER_TERM)).copy().reset_index(drop=True)

if len(finalists) == 0:
    raise ValueError("No finalist settings found from quick report.")

print("Refine finalists selected:")
print(finalists.groupby("term").size().to_dict())
display(finalists[[
    "term", "subset", "k_latents", "alpha", "direction", "edit_first_k",
    "retrieval_score", "top_minus_random_delta_p", "top_minus_random_mean_delta_retr_top",
]].head(20))

rows = []
start = time.time()
for i, r in finalists.iterrows():
    term = str(r["term"])
    if term not in primary_contexts:
        continue

    ctx = primary_contexts[term]
    subsets_all = select_eval_subsets(
        ctx,
        q=RETRIEVAL_IMPROVEMENT_CONFIG["near_boundary_quantile"],
        min_n=RETRIEVAL_IMPROVEMENT_CONFIG["near_boundary_min_n"],
    )

    subset_name = str(r["subset"])
    if subset_name not in subsets_all:
        continue

    row = evaluate_retrieval_setting(
        ctx=ctx,
        subset_name=subset_name,
        subset_df=subsets_all[subset_name],
        k_latents=int(r["k_latents"]),
        alpha=float(r["alpha"]),
        direction=int(r["direction"]),
        edit_first_k=int(r["edit_first_k"]),
        top_n=RETRIEVAL_IMPROVEMENT_CONFIG["top_n_retrieval"],
        random_trials=REFINE_CONFIG["random_trials"],
        shuffled_trials=REFINE_CONFIG["shuffled_trials"],
        bootstrap_n=REFINE_CONFIG["bootstrap_n"],
        seed=REFINE_CONFIG["seed"],
    )
    rows.append(row)

    done = i + 1
    if done % 4 == 0 or done == len(finalists):
        elapsed = max(1e-6, time.time() - start)
        rate = done / elapsed
        rem = len(finalists) - done
        eta_min = rem / max(rate, 1e-6) / 60.0
        print(f"Refine progress: {done}/{len(finalists)} | ETA={eta_min:.1f}m")

refine_df = pd.DataFrame(rows)
if len(refine_df) == 0:
    raise ValueError("Refine pass produced no rows.")

refine_df["retrieval_score"] = (
    2.5 * refine_df["top_minus_random_mean_delta_retr_top"].fillna(0.0)
    + 1.5 * refine_df["top_minus_signflip_mean_delta_retr_top"].fillna(0.0)
    + 0.8 * refine_df["top_minus_random_mean_new_positive_count_top"].fillna(0.0)
    + 0.6 * refine_df["top_minus_random_mean_new_neighbor_label_top"].fillna(0.0)
    + 0.8 * refine_df["top_minus_random_mean_weighted_knn_label_shift_top"].fillna(0.0)
    + 0.9 * refine_df["bootstrap_top_minus_random_delta_retr_ci_lo"].fillna(0.0)
    + 0.5 * refine_df["bootstrap_top_minus_signflip_delta_retr_ci_lo"].fillna(0.0)
    + 0.3 * refine_df["shuffled_retr_delta_z"].fillna(0.0)
    + 0.05 * refine_df["top_minus_random_delta_p"].fillna(0.0)
)

refine_df = refine_df.sort_values(["term", "retrieval_score", "top_minus_random_delta_p"], ascending=[True, False, False]).reset_index(drop=True)
refine_df["run_mode"] = "refine"
refine_df["refine_top_k_per_term"] = int(REFINE_TOP_K_PER_TERM)
refine_df["refine_random_trials"] = int(REFINE_CONFIG["random_trials"])
refine_df["refine_shuffled_trials"] = int(REFINE_CONFIG["shuffled_trials"])
refine_df["refine_bootstrap_n"] = int(REFINE_CONFIG["bootstrap_n"])
refine_df["rank_within_term"] = refine_df.groupby("term").cumcount() + 1
refine_df["is_term_best"] = refine_df["rank_within_term"] == 1

refine_df.to_csv(REFINE_REPORT_PATH, index=False)
print(f"Saved {REFINE_REPORT_PATH}")
print(f"Refine elapsed: {(time.time() - start)/60.0:.1f} min")

summary_cols = [
    "term", "subset", "k_latents", "alpha", "direction", "edit_first_k",
    "top_minus_random_mean_delta_retr_top", "bootstrap_top_minus_random_delta_retr_ci_lo",
    "top_minus_signflip_mean_delta_retr_top", "shuffled_retr_p_ge_obs",
    "top_minus_random_delta_p", "retrieval_score", "rank_within_term", "is_term_best",
]
summary_cols = [c for c in summary_cols if c in refine_df.columns]
display(refine_df[summary_cols].head(20))

In [ ]:
# Step 13F: disorder-only mini-refine (top finalists, stronger uncertainty)

import time

DISORDER_MINI_REFINE_TOP_K = 3
DISORDER_MINI_REFINE_CONFIG = {
    "random_trials": 12,
    "shuffled_trials": 60,
    "bootstrap_n": 2000,
    "seed": int(RETRIEVAL_IMPROVEMENT_CONFIG["seed"]),
}

DISORDER_MINI_REFINE_PATH = os.path.join(RETRIEVAL_OUTPUT_DIR, "retrieval_improvement_report_disorder_mini_refine.csv")

source_candidates = [
    os.path.join(RETRIEVAL_OUTPUT_DIR, "retrieval_improvement_report_refine.csv"),
    os.path.join(RETRIEVAL_OUTPUT_DIR, "retrieval_improvement_report.csv"),
]
source_path = None
for p in source_candidates:
    if os.path.exists(p):
        source_path = p
        break
if source_path is None:
    raise FileNotFoundError("Need retrieval_improvement_report_refine.csv or retrieval_improvement_report.csv before Step 13F.")

base_df = pd.read_csv(source_path)
base_df = base_df[base_df["term"] == "disorder"].copy()
if len(base_df) == 0:
    raise ValueError("No disorder rows found in source report.")

if "retrieval_score" not in base_df.columns:
    raise ValueError("Source report missing retrieval_score.")

finalists = (
    base_df.sort_values(["retrieval_score", "top_minus_random_delta_p"], ascending=[False, False])
    .head(int(DISORDER_MINI_REFINE_TOP_K))
    .reset_index(drop=True)
)

print(f"Disorder mini-refine source: {source_path}")
print(f"Selected top {len(finalists)} disorder settings")
display(finalists[[
    "term", "subset", "k_latents", "alpha", "direction", "edit_first_k",
    "retrieval_score", "top_minus_random_delta_p", "top_minus_random_mean_delta_retr_top",
]])

if "disorder" not in primary_contexts:
    raise ValueError("disorder context missing; run Step 12B first.")

ctx = primary_contexts["disorder"]
subsets_all = select_eval_subsets(
    ctx,
    q=RETRIEVAL_IMPROVEMENT_CONFIG["near_boundary_quantile"],
    min_n=RETRIEVAL_IMPROVEMENT_CONFIG["near_boundary_min_n"],
)

rows = []
start = time.time()
for i, r in finalists.iterrows():
    subset_name = str(r["subset"])
    if subset_name not in subsets_all:
        continue

    out_row = evaluate_retrieval_setting(
        ctx=ctx,
        subset_name=subset_name,
        subset_df=subsets_all[subset_name],
        k_latents=int(r["k_latents"]),
        alpha=float(r["alpha"]),
        direction=int(r["direction"]),
        edit_first_k=int(r["edit_first_k"]),
        top_n=RETRIEVAL_IMPROVEMENT_CONFIG["top_n_retrieval"],
        random_trials=DISORDER_MINI_REFINE_CONFIG["random_trials"],
        shuffled_trials=DISORDER_MINI_REFINE_CONFIG["shuffled_trials"],
        bootstrap_n=DISORDER_MINI_REFINE_CONFIG["bootstrap_n"],
        seed=DISORDER_MINI_REFINE_CONFIG["seed"],
    )
    rows.append(out_row)

    done = i + 1
    elapsed = max(1e-6, time.time() - start)
    eta_min = (len(finalists) - done) / max(done / elapsed, 1e-6) / 60.0
    print(f"Mini-refine progress: {done}/{len(finalists)} | ETA={eta_min:.1f}m")

mini_df = pd.DataFrame(rows)
if len(mini_df) == 0:
    raise ValueError("Mini-refine produced no rows.")

mini_df["retrieval_score"] = (
    2.5 * mini_df["top_minus_random_mean_delta_retr_top"].fillna(0.0)
    + 1.5 * mini_df["top_minus_signflip_mean_delta_retr_top"].fillna(0.0)
    + 0.8 * mini_df["top_minus_random_mean_new_positive_count_top"].fillna(0.0)
    + 0.6 * mini_df["top_minus_random_mean_new_neighbor_label_top"].fillna(0.0)
    + 0.8 * mini_df["top_minus_random_mean_weighted_knn_label_shift_top"].fillna(0.0)
    + 0.9 * mini_df["bootstrap_top_minus_random_delta_retr_ci_lo"].fillna(0.0)
    + 0.5 * mini_df["bootstrap_top_minus_signflip_delta_retr_ci_lo"].fillna(0.0)
    + 0.3 * mini_df["shuffled_retr_delta_z"].fillna(0.0)
    + 0.05 * mini_df["top_minus_random_delta_p"].fillna(0.0)
)
mini_df = mini_df.sort_values(["retrieval_score", "top_minus_random_delta_p"], ascending=[False, False]).reset_index(drop=True)
mini_df["run_mode"] = "disorder_mini_refine"
mini_df["mini_refine_top_k"] = int(DISORDER_MINI_REFINE_TOP_K)
mini_df["mini_refine_random_trials"] = int(DISORDER_MINI_REFINE_CONFIG["random_trials"])
mini_df["mini_refine_shuffled_trials"] = int(DISORDER_MINI_REFINE_CONFIG["shuffled_trials"])
mini_df["mini_refine_bootstrap_n"] = int(DISORDER_MINI_REFINE_CONFIG["bootstrap_n"])
mini_df["rank_within_disorder"] = np.arange(1, len(mini_df) + 1)
mini_df["is_best_disorder"] = mini_df["rank_within_disorder"] == 1
mini_df["retrieval_ci_positive"] = mini_df["bootstrap_top_minus_random_delta_retr_ci_lo"] > 0

mini_df.to_csv(DISORDER_MINI_REFINE_PATH, index=False)
print(f"Saved {DISORDER_MINI_REFINE_PATH}")
print(f"Mini-refine elapsed: {(time.time() - start)/60.0:.1f} min")

summary_cols = [
    "subset", "k_latents", "alpha", "direction", "edit_first_k",
    "top_minus_random_mean_delta_retr_top",
    "bootstrap_top_minus_random_delta_retr_ci_lo",
    "bootstrap_top_minus_random_delta_retr_ci_hi",
    "retrieval_ci_positive",
    "top_minus_random_delta_p",
    "shuffled_retr_p_ge_obs",
    "retrieval_score",
    "rank_within_disorder",
    "is_best_disorder",
]
summary_cols = [c for c in summary_cols if c in mini_df.columns]
display(mini_df[summary_cols])

best = mini_df.iloc[0]
if bool(best.get("retrieval_ci_positive", False)):
    print("Disorder retrieval evidence strengthened: best top-vs-random CI lower bound is > 0.")
else:
    print("Disorder still borderline: best top-vs-random CI lower bound is <= 0.")

## Extension: `disorder_to_order` — diagnostics, stricter probe, Step-12-tuned retrieval, reversibility

**New cells only** (nothing above was modified). Prerequisites: through **Step 4A** (SAE), **Step 8A**, **Step 8C**, **Step 11C** (ranking cache), **Step 9A** (strict helpers), **Step 12B–12C** (`primary_terms_best_settings.csv`). **Step 28** defines `seq_to_orig_emb_term` / `seq_to_edit_emb_term`; if you skipped it, this extension defines fallbacks inside each code cell.


In [ ]:
# Extension A1: diagnostics — disorder vs disorder_to_order

import os
import numpy as np
import pandas as pd

def _summarize_labels(path, term_hint=None):
    if not os.path.exists(path):
        print(f"missing {path}")
        return
    df = pd.read_csv(path)
    if term_hint and "term" in df.columns:
        df = df[df["term"] == term_hint]
    print(f"\n=== {path} ===")
    if len(df) == 0:
        print("empty")
        return
    for sp in sorted(df["split"].unique()):
        sub = df[df["split"] == sp]
        vc = sub["protein_label"].value_counts(dropna=False).sort_index()
        print(f"split={sp} protein_label counts:\n{vc}")
    if "protein_score" in df.columns:
        print("protein_score describe:\n", df["protein_score"].describe())

_summarize_labels("labels_disorder.csv", term_hint="disorder")
_summarize_labels("labels_disorder_to_order.csv", term_hint="disorder_to_order")

for name in ["probe_metrics_disorder.csv", "probe_metrics_disorder_to_order.csv"]:
    if os.path.exists(name):
        m = pd.read_csv(name)
        t = m.pivot(index="term", columns="split", values=["auc", "f1", "acc"])
        print(f"\n=== {name} (pivot) ===")
        display(t)
    else:
        print(f"missing {name}")


In [ ]:
# Extension A2: stricter labels + threshold-tuned probe for disorder_to_order; pick best vs baseline

import os
import numpy as np
import pandas as pd

TERM_D2O = "disorder_to_order"
LABELS_BASE = "labels_disorder_to_order.csv"

clf_disorder_to_order_active = None
disorder_to_order_probe_threshold = 0.5
disorder_to_order_probe_source = "unset"
D2O_RANK_NPZ = "latent_rank_disorder_to_order.npz"

baseline_auc_test = np.nan
if os.path.exists("probe_metrics_disorder_to_order.csv"):
    pm = pd.read_csv("probe_metrics_disorder_to_order.csv")
    te = pm[pm["split"] == "test"]
    if len(te):
        baseline_auc_test = float(te["auc"].iloc[0])

strict_rows = []
best_strict = None
best_val_auc = -1.0
best_q = None
best_clf = None
best_metrics_df = None

for q in [0.50, 0.70, 0.85]:
    try:
        strict_df, cutoff = build_strict_labels_from_scores(
            TERM_D2O, labels_csv=LABELS_BASE, pos_quantile_nonzero=q
        )
        clf_s, met_s = train_with_threshold_tuning(TERM_D2O, strict_df)
        val_auc = float(met_s[met_s["split"] == "val"]["auc"].iloc[0])
        test_auc = float(met_s[met_s["split"] == "test"]["auc"].iloc[0])
        strict_rows.append({"q": q, "cutoff": cutoff, "val_auc": val_auc, "test_auc": test_auc})
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_q = q
            best_clf = clf_s
            best_metrics_df = met_s
    except Exception as ex:
        strict_rows.append({"q": q, "cutoff": np.nan, "val_auc": np.nan, "test_auc": np.nan, "error": str(ex)})

scan_df = pd.DataFrame(strict_rows)
print("Strict-label scan (disorder_to_order):")
display(scan_df)

if best_clf is not None and best_metrics_df is not None:
    strict_test_auc = float(best_metrics_df[best_metrics_df["split"] == "test"]["auc"].iloc[0])
    thr_row = best_metrics_df[best_metrics_df["split"] == "val"].iloc[0]
    thr = float(thr_row["threshold"])
    if strict_test_auc > baseline_auc_test:
        clf_disorder_to_order_active = best_clf
        disorder_to_order_probe_threshold = thr
        disorder_to_order_probe_source = f"strict_q{best_q}_thr_tuned"
        print(f"Using STRICT probe (best q={best_q}, val_thr={thr:.3f}): test_auc={strict_test_auc:.4f} > baseline {baseline_auc_test:.4f}")
    else:
        clf_b, _ = train_probe_for_term(TERM_D2O, LABELS_BASE)
        clf_disorder_to_order_active = clf_b
        disorder_to_order_probe_threshold = 0.5
        disorder_to_order_probe_source = "baseline_step8c"
        print(f"Keeping BASELINE probe: strict test_auc={strict_test_auc:.4f} <= baseline {baseline_auc_test:.4f}")
    best_metrics_df.to_csv("probe_metrics_disorder_to_order_strict_best.csv", index=False)
    print("Saved probe_metrics_disorder_to_order_strict_best.csv")
else:
    clf_b, _ = train_probe_for_term(TERM_D2O, LABELS_BASE)
    clf_disorder_to_order_active = clf_b
    disorder_to_order_probe_source = "baseline_step8c_fallback"
    print("Strict training failed for all q; using baseline probe.")

scan_df.to_csv("disorder_to_order_strict_label_scan.csv", index=False)


In [ ]:
# Extension A2b (optional): alternate ranking cache with token-mean pooling (writes separate .npz)

import os
import numpy as np
import pandas as pd
import torch

D2O_RECOMPUTE_RANK_WITH_MEAN = False

if not D2O_RECOMPUTE_RANK_WITH_MEAN:
    print("Set D2O_RECOMPUTE_RANK_WITH_MEAN = True to write latent_rank_disorder_to_order_mean.npz")
else:
    manifest = pd.read_csv("protein_manifest_idpinterp.csv")
    labels_df = pd.read_csv("labels_disorder_to_order.csv")
    train_labeled = manifest.merge(labels_df[["acc", "protein_label"]], on="acc", how="inner")
    train_labeled = train_labeled[
        (train_labeled["split"] == "train") & (train_labeled["protein_label"].isin([0, 1]))
    ].copy().reset_index(drop=True)

    @torch.no_grad()
    def _prot_latent_mean(seq):
        h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
        h = h[:K_TOKENS]
        if h.shape[0] == 0:
            return np.zeros(sae.enc.out_features, dtype=np.float32)
        z = torch.relu(sae.enc(h))
        Z = z.mean(dim=0).values
        return Z.detach().cpu().numpy().astype(np.float32)

    M = min(800, len(train_labeled))
    seqsM = [str(s)[:SEQ_TRUNC] for s in train_labeled["sequence"].tolist()[:M]]
    yM = train_labeled["protein_label"].to_numpy(dtype=np.float32)[:M]
    Zmat = np.stack([_prot_latent_mean(s) for s in seqsM], axis=0)
    ranked_latents, corr = rank_latents_by_correlation(Zmat, yM)
    top1_sd = Zmat[:, int(ranked_latents[0])].std() + 1e-8
    alphas_local = np.array([0, 1 * top1_sd, 2 * top1_sd, 4 * top1_sd], dtype=np.float32)
    out_path = "latent_rank_disorder_to_order_mean.npz"
    np.savez(out_path, ranked_latents=ranked_latents, corr=corr, alphas=alphas_local)
    print("Saved", out_path)
    D2O_RANK_NPZ = out_path


In [ ]:
# Extension A3: retrieval + probe deltas using Step 12 best settings for disorder_to_order

import json
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity

TERM_D2O = "disorder_to_order"
LABELS_PATH = "labels_disorder_to_order.csv"

if "clf_disorder_to_order_active" not in dir() or clf_disorder_to_order_active is None:
    clf_disorder_to_order_active, _ = train_probe_for_term(TERM_D2O, LABELS_PATH)

rank_path = "latent_rank_disorder_to_order_mean.npz" if os.path.exists("latent_rank_disorder_to_order_mean.npz") else "latent_rank_disorder_to_order.npz"
rc = np.load(rank_path)
ranked_latents = rc["ranked_latents"]

@torch.no_grad()
def _seq_to_orig_emb_d2o(seq):
    h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
    h = h[:K_TOKENS]
    if h.shape[0] == 0:
        return np.zeros(480, dtype=np.float32)
    return h.mean(dim=0).detach().cpu().numpy().astype(np.float32)

@torch.no_grad()
def _seq_to_edit_emb_d2o(seq, latent_indices, alpha, edit_first_k):
    h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
    h = h[:K_TOKENS]
    if h.shape[0] == 0:
        return _seq_to_orig_emb_d2o(seq)
    z = torch.relu(sae.enc(h))
    z_edit = z.clone()
    k = min(int(edit_first_k), z_edit.shape[0])
    z_edit[:k, latent_indices] += float(alpha)
    h_hat = sae.dec(z_edit)
    return h_hat.mean(dim=0).detach().cpu().numpy().astype(np.float32)

def _prob_d2o(x):
    return float(clf_disorder_to_order_active.predict_proba(x.reshape(1, -1))[0, 1])

manifest = pd.read_csv("protein_manifest_idpinterp.csv")
labels_df = pd.read_csv(LABELS_PATH)
emb = np.load("embeddings_idpinterp.npz", allow_pickle=True)
embed_df = pd.DataFrame({"acc": emb["accs"].astype(str)})
embed_df["embedding"] = list(emb["X"])

meta = manifest[["acc", "split"]].merge(labels_df[["acc", "protein_label"]], on="acc", how="inner")
meta = meta.merge(embed_df, on="acc", how="inner")
meta = meta[meta["protein_label"].isin([0, 1])].copy()
train_bank = meta[meta["split"] == "train"].copy().reset_index(drop=True)
test_bank = meta[meta["split"] == "test"].copy().reset_index(drop=True)
bank_emb = np.stack(train_bank["embedding"].to_list(), axis=0)
bank_y = train_bank["protein_label"].astype(int).to_numpy()
seq_map = dict(zip(manifest["acc"].astype(str), manifest["sequence"].astype(str)))

best_k, best_alpha, best_direction, best_efk = 5, 0.2, 1, 30
src = "default_fallback"
if os.path.exists("primary_terms_best_settings.csv"):
    bs = pd.read_csv("primary_terms_best_settings.csv")
    row = bs[bs["term"] == TERM_D2O]
    if len(row):
        r0 = row.iloc[0]
        best_k = int(r0["k_latents"])
        best_alpha = float(r0["alpha"])
        best_direction = int(r0["direction"])
        best_efk = int(r0["edit_first_k"])
        src = "primary_terms_best_settings.csv"
elif os.path.exists(f"edit_compare_top_vs_random_{TERM_D2O}_pivot.csv"):
    pv = pd.read_csv(f"edit_compare_top_vs_random_{TERM_D2O}_pivot.csv")
    if len(pv):
        j = int(pv["top_minus_random"].values.argmax())
        r0 = pv.iloc[j]
        best_k = int(r0["k_latents"])
        best_alpha = float(r0["alpha"])
        best_direction = int(r0["direction"])
        best_efk = 30
        src = "edit_compare_pivot"

alpha_eff = float(best_alpha) * int(best_direction)
lat_idx = ranked_latents[:best_k]

cfg = {
    "term": TERM_D2O,
    "rank_npz": rank_path,
    "settings_source": src,
    "k_latents": best_k,
    "alpha": best_alpha,
    "direction": best_direction,
    "edit_first_k": best_efk,
    "alpha_eff": alpha_eff,
}
with open("disorder_to_order_active_edit_config.json", "w") as f:
    json.dump(cfg, f, indent=2)
print("Active edit config:", cfg)

TOP_N = 10
rows = []
for _, r in test_bank.head(min(100, len(test_bank))).iterrows():
    acc = str(r["acc"])
    seq = str(seq_map[acc])[:SEQ_TRUNC]
    y_true = int(r["protein_label"])
    x0 = _seq_to_orig_emb_d2o(seq)
    x1 = _seq_to_edit_emb_d2o(seq, lat_idx, alpha_eff, best_efk)
    p0, p1 = _prob_d2o(x0), _prob_d2o(x1)
    sims0 = cosine_similarity(x0.reshape(1, -1), bank_emb)[0]
    sims1 = cosine_similarity(x1.reshape(1, -1), bank_emb)[0]
    i0 = np.argsort(-sims0)[:TOP_N]
    i1 = np.argsort(-sims1)[:TOP_N]
    rows.append({
        "acc": acc,
        "y_true": y_true,
        "p_orig": p0,
        "p_edit": p1,
        "delta_p": p1 - p0,
        "orig_neighbor_pos_rate": float(bank_y[i0].mean()),
        "edit_neighbor_pos_rate": float(bank_y[i1].mean()),
        "delta_neighbor_pos_rate": float(bank_y[i1].mean() - bank_y[i0].mean()),
        "mean_sim_orig": float(np.mean(sims0[i0])),
        "mean_sim_edit": float(np.mean(sims1[i1])),
    })

ret_tune = pd.DataFrame(rows)
ret_tune.to_csv("retrieval_shift_disorder_to_order_tuned.csv", index=False)
print("Saved retrieval_shift_disorder_to_order_tuned.csv")
display(ret_tune.describe())


### Reversibility experiment (`disorder_to_order` only)

Forward: add `alpha_eff` on top-ranked latents (first `edit_first_k` tokens). Reverse: subtract the same amount on the **forward** activations so the code returns to `z` before decode. Compare **random** latent control. Uses `clf_disorder_to_order_active` and `disorder_to_order_active_edit_config.json` if present.


In [ ]:
# Extension B: reversibility batch metrics (probe, embedding, retrieval)

import json
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity

TERM_D2O = "disorder_to_order"
LABELS_PATH = "labels_disorder_to_order.csv"
EPSILON = 0.02
N_EVAL = 100
TOP_N = 10
RNG = np.random.default_rng(int(SEED) + 901)

if "clf_disorder_to_order_active" not in dir() or clf_disorder_to_order_active is None:
    clf_disorder_to_order_active, _ = train_probe_for_term(TERM_D2O, LABELS_PATH)

cfg_path = "disorder_to_order_active_edit_config.json"
if os.path.exists(cfg_path):
    cfg = json.load(open(cfg_path))
    rank_path = cfg.get("rank_npz", "latent_rank_disorder_to_order.npz")
    best_k = int(cfg["k_latents"])
    best_alpha = float(cfg["alpha"])
    best_direction = int(cfg["direction"])
    best_efk = int(cfg["edit_first_k"])
    alpha_eff = float(cfg["alpha_eff"])
else:
    rank_path = "latent_rank_disorder_to_order_mean.npz" if os.path.exists("latent_rank_disorder_to_order_mean.npz") else "latent_rank_disorder_to_order.npz"
    rc = np.load(rank_path)
    best_k, best_alpha, best_direction, best_efk = 5, 0.2, 1, 30
    alpha_eff = float(best_alpha) * int(best_direction)

rc = np.load(rank_path)
ranked_latents = rc["ranked_latents"]
lat_top = ranked_latents[:best_k]
d_lat = int(sae.enc.out_features)

manifest = pd.read_csv("protein_manifest_idpinterp.csv")
labels_df = pd.read_csv(LABELS_PATH)
emb = np.load("embeddings_idpinterp.npz", allow_pickle=True)
embed_df = pd.DataFrame({"acc": emb["accs"].astype(str)})
embed_df["embedding"] = list(emb["X"])
meta = manifest[["acc", "split"]].merge(labels_df[["acc", "protein_label"]], on="acc", how="inner")
meta = meta.merge(embed_df, on="acc", how="inner")
meta = meta[meta["protein_label"].isin([0, 1])].copy()
train_bank = meta[meta["split"] == "train"].copy().reset_index(drop=True)
bank_emb = np.stack(train_bank["embedding"].to_list(), axis=0)
bank_y = train_bank["protein_label"].astype(int).to_numpy()
seq_map = dict(zip(manifest["acc"].astype(str), manifest["sequence"].astype(str)))
test_bank = meta[meta["split"] == "test"].copy().reset_index(drop=True)

def _prob(x):
    return float(clf_disorder_to_order_active.predict_proba(x.reshape(1, -1))[0, 1])

def _cos(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return np.nan
    return float(np.dot(a, b) / (na * nb))

def _jacc(i0, i1):
    a = set(np.asarray(i0).tolist())
    b = set(np.asarray(i1).tolist())
    u = a | b
    return float(len(a & b) / len(u)) if len(u) else np.nan

def _nn(xq):
    sims = cosine_similarity(xq.reshape(1, -1), bank_emb)[0]
    idx = np.argsort(-sims)[:TOP_N]
    return idx, sims[idx]

@torch.no_grad()
def _forward_undo(seq, latent_indices, alpha_eff, edit_first_k, rng_latents=None):
    h = get_token_hidden(seq, tok, model, max_length=MAXLEN, drop_special=True)
    h = h[:K_TOKENS]
    if h.shape[0] == 0:
        z0 = torch.zeros(1, 1, device=h.device)
        x0 = np.zeros(480, dtype=np.float32)
        return x0, x0, x0, x0, x0, x0, x0, x0
    x0 = h.mean(dim=0).detach().cpu().numpy().astype(np.float32)
    z = torch.relu(sae.enc(h))
    z0 = z.clone()
    k = min(int(edit_first_k), z0.shape[0])
    idx = np.asarray(latent_indices, dtype=np.int64)
    h_rec = sae.dec(z0)
    x_recon = h_rec.mean(dim=0).detach().cpu().numpy().astype(np.float32)
    zf = z0.clone()
    if rng_latents is None:
        zf[:k, idx] += float(alpha_eff)
    else:
        zf[:k, rng_latents] += float(alpha_eff)
    xf = sae.dec(zf).mean(dim=0).detach().cpu().numpy().astype(np.float32)
    zr = zf.clone()
    if rng_latents is None:
        zr[:k, idx] -= float(alpha_eff)
    else:
        zr[:k, rng_latents] -= float(alpha_eff)
    xr = sae.dec(zr).mean(dim=0).detach().cpu().numpy().astype(np.float32)
    return x0, x_recon, xf, xr, x0, x0, x0, x0

rows = []
for _, r in test_bank.head(min(N_EVAL, len(test_bank))).iterrows():
    acc = str(r["acc"])
    seq = str(seq_map[acc])[:SEQ_TRUNC]
    lat_rand = RNG.choice(d_lat, size=best_k, replace=False)

    x0, x_rec, xf_t, xr_t, *_ = _forward_undo(seq, lat_top, alpha_eff, best_efk, None)
    _, _, xf_r, xr_r, *_ = _forward_undo(seq, lat_rand, alpha_eff, best_efk, lat_rand)

    s0 = _prob(x0)
    s0_rec = _prob(x_rec)
    s1t, s2t = _prob(xf_t), _prob(xr_t)
    s1r, s2r = _prob(xf_r), _prob(xr_r)

    i0, _ = _nn(x0)
    i_f_t, _ = _nn(xf_t)
    i_r_t, _ = _nn(xr_t)
    i_f_r, _ = _nn(xf_r)
    i_r_r, _ = _nn(xr_r)

    rows.append({
        "acc": acc,
        "mode_top": 1,
        "s0": s0,
        "s0_recon": s0_rec,
        "s1": s1t,
        "s2": s2t,
        "forward_shift": s1t - s0,
        "reverse_shift": s2t - s1t,
        "recovery_gap": abs(s2t - s0),
        "recovery_gap_recon": abs(s2t - s0_rec),
        "recovered_eps": int(abs(s2t - s0) < EPSILON),
        "recovered_recon_eps": int(abs(s2t - s0_rec) < EPSILON),
        "cos_orig_fwd": _cos(x0, xf_t),
        "cos_fwd_rev": _cos(xf_t, xr_t),
        "cos_orig_rev": _cos(x0, xr_t),
        "cos_recon_rev": _cos(x_rec, xr_t),
        "jacc_orig_fwd": _jacc(i0, i_f_t),
        "jacc_orig_rev": _jacc(i0, i_r_t),
        "rand_s1": s1r,
        "rand_s2": s2r,
        "rand_recovery_gap": abs(s2r - s0),
        "rand_recovery_gap_recon": abs(s2r - s0_rec),
        "rand_jacc_orig_fwd": _jacc(i0, i_f_r),
        "rand_jacc_orig_rev": _jacc(i0, i_r_r),
    })

rev_df = pd.DataFrame(rows)
rev_df.to_csv("reversibility_disorder_to_order.csv", index=False)
print("Saved reversibility_disorder_to_order.csv (epsilon=%.3f)" % EPSILON)

agg = {
    "mean_forward_shift_top": rev_df["forward_shift"].mean(),
    "mean_reverse_shift_top": rev_df["reverse_shift"].mean(),
    "mean_recovery_gap_top": rev_df["recovery_gap"].mean(),
    "mean_recovery_gap_recon_top": rev_df["recovery_gap_recon"].mean(),
    "pct_recovered_eps_top": rev_df["recovered_eps"].mean(),
    "pct_recovered_recon_eps_top": rev_df["recovered_recon_eps"].mean(),
    "mean_jacc_orig_fwd_top": rev_df["jacc_orig_fwd"].mean(),
    "mean_jacc_orig_rev_top": rev_df["jacc_orig_rev"].mean(),
    "mean_rand_recovery_gap": rev_df["rand_recovery_gap"].mean(),
    "mean_rand_recovery_recon": rev_df["rand_recovery_gap_recon"].mean(),
    "mean_rand_jacc_orig_rev": rev_df["rand_jacc_orig_rev"].mean(),
}
print(pd.Series(agg))
display(rev_df.head())
